# Create Knowledge Probes
This generates our knowledge probes to test for factual recall.

In [1]:
import textwrap
import sys 
sys.path.append('../..')
import utils.utils as utils
import pandas as pd

with open('../../data/arxiv/cleaned_DPO.txt', 'r') as f:
    paper = f.read()


def print_wrapped(text, width=100):
    """
    Prints the given text wrapped to a specified width for better readability in notebooks.
    This function preserves paragraph breaks.
    """
    paragraphs = text.split('\n\n')
    for para in paragraphs:
        print(textwrap.fill(para, width=width))
        print()
        
print_wrapped(paper)

\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

\begin{abstract} While large-scale unsupervised language models (LMs) learn broad world knowledge
and some reasoning skills, achieving precise control of their behavior is difficult due to the
completely unsupervised nature of their training. Existing methods for gaining such steerability
collect human labels of the relative quality of model generations and fine-tune the unsupervised LM
to align with these preferences, often with reinforcement learning from human feedback (RLHF).
However, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects
the human preferences, and then fine-tuning the large unsupervised LM using reinforcement learning
to maximize this estimated reward without drifting too far from the original model. In this paper we
introduce a new parameterization of the reward model in RLHF that enables extraction of the
corresponding optimal policy in clo

\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

\begin{abstract} While large-scale unsupervised language models (LMs) learn broad world knowledge
and some reasoning skills, achieving precise control of their behavior is difficult due to the
completely unsupervised nature of their training. Existing methods for gaining such steerability
collect human labels of the relative quality of model generations and fine-tune the unsupervised LM
to align with these preferences, often with reinforcement learning from human feedback (RLHF).
However, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects
the human preferences, and then fine-tuning the large unsupervised LM using reinforcement learning
to maximize this estimated reward without drifting too far from the original model. In this paper we
introduce a new parameterization of the reward model in RLHF that enables extraction of the
corresponding optimal policy in closed form, allowing us to solve the standard RLHF problem with
only a simple classification loss. The resulting algorithm, which we call \textit{Direct Preference
Optimization} (DPO), is stable, performant, and computationally lightweight, eliminating the need
for sampling from the LM during fine-tuning or performing significant hyperparameter tuning. Our
experiments show that DPO can fine-tune LMs to align with human preferences as well as or better
than existing methods. Notably, fine-tuning with DPO exceeds PPO-based RLHF in ability to control
sentiment of generations, and matches or improves response quality in summarization and single-turn
dialogue while being substantially simpler to implement and train. \end{abstract}

\section{Introduction} Large unsupervised language models (LMs) trained on very large datasets
acquire surprising capabilities~\citep{chowdhery2022palm, brown2020language,
touvron2023llama,bubeck2023sparks}. However, these models are trained on data generated by humans
with a wide variety of goals, priorities, and skillsets. Some of these goals and skillsets may not
be desirable to imitate; for example, while we may want our AI coding assistant to
\textit{understand} common programming mistakes in order to correct them, nevertheless, when
generating code, we would like to bias our model toward the (potentially rare) high-quality coding
ability present in its training data. Similarly, we might want our language model to be
\textit{aware} of a common misconception believed by 50\% of people, but we certainly do not want
the model to claim this misconception to be true in 50\% of queries about it! In other words,
selecting the model's \emph{desired responses and behavior} from its very wide \textit{knowledge and
abilities} is crucial to building AI systems that are safe, performant, and controllable
\citep{ouyang2022training}. While existing methods typically steer LMs to match human preferences
using reinforcement learning (RL), we will show that the RL-based objective used by existing methods
can be optimized exactly with a simple binary cross-entropy objective, greatly simplifying the
preference learning pipeline.

\begin{figure}     \centering
\includegraphics[width=0.999\textwidth]{figures/diagrams/teaser.png}     \caption{\textbf{DPO
optimizes for human preferences while avoiding reinforcement learning.} Existing methods for fine-
tuning language models with human feedback first fit a reward model to a dataset of prompts and
human preferences over pairs of responses, and then use RL to find a policy that maximizes the
learned reward. In contrast, DPO directly optimizes for the policy best satisfying the preferences
with a simple classification objective, fitting an \textit{implicit} reward model whose
corresponding optimal policy can be extracted in closed form.}     \vspace{-2mm}
\label{fig:teaser} \end{figure}

At a high level, existing methods instill the desired behaviors into a language model using curated
sets of human preferences representing the types of behaviors that humans find safe and helpful.
This preference learning stage occurs after an initial stage of large-scale unsupervised pre-
training on a large text dataset. While the most straightforward approach to preference learning is
supervised fine-tuning on human demonstrations of high quality responses, the most successful class
of methods is reinforcement learning from human (or AI) feedback (RLHF/RLAIF;
\citep{christiano2017deep,bai2022constitutional}). RLHF methods fit a reward model to a dataset of
human preferences and then use RL to optimize a language model policy to produce responses assigned
high reward without drifting excessively far from the original model. While RLHF produces models
with impressive conversational and coding abilities, the RLHF pipeline is considerably more complex
than supervised learning, involving training multiple LMs and sampling from the LM policy in the
loop of training, incurring significant computational costs.

In this paper, we show how to directly optimize a language model to adhere to human preferences,
without explicit reward modeling or reinforcement learning. We propose Direct Preference
Optimization (DPO), an algorithm that implicitly optimizes the same objective as existing RLHF
algorithms (reward maximization with a KL-divergence constraint) but is simple to implement and
straightforward to train. Intuitively, the DPO update increases the relative log probability of
preferred to dispreferred responses, but it incorporates a dynamic, per-example importance weight
that prevents the model degeneration that we find occurs with a naive probability ratio objective.
Like existing algorithms, DPO relies on a theoretical preference model (such as the Bradley-Terry
model; \cite{bradley1952rankanalysis}) that measures how well a given reward function aligns with
empirical preference data. However, while existing methods use the preference model to define a
preference loss to train a reward model and then train a policy that optimizes the learned reward
model, DPO uses a change of variables to define the preference loss as a function of the policy
directly. Given a dataset of human preferences over model responses, DPO can therefore optimize a
policy using a simple binary cross entropy objective, producing the optimal policy to an implicit
reward function fit to the preference data.

Our main contribution is Direct Preference Optimization (DPO), a simple RL-free algorithm for
training language models from preferences. Our experiments show that DPO is at least as effective as
existing methods, including PPO-based RLHF, for learning from preferences in tasks such as sentiment
modulation, summarization, and dialogue, using language models with up to 6B parameters.

\section{Related Work}

Self-supervised language models of increasing scale learn to complete some tasks zero-shot
\citep{radford2019language} or with few-shot prompts \citep{gpt3,megatron,chowdhery2022palm}.
However, their performance on downstream tasks and alignment with user intent can be significantly
improved by fine-tuning on datasets of instructions and human-written completions \citep{mishra-
etal-2022-cross,sanh2022multitask,chung2022scaling,thoppilan2022lamda}. This `instruction-tuning'
procedure enables LLMs to generalize to instructions outside of the instruction-tuning set and
generally increase their usability \citep{chung2022scaling}. Despite the success of instruction
tuning, \textit{relative} human judgments of response quality are often easier to collect than
expert demonstrations, and thus subsequent works have fine-tuned LLMs with datasets of human
preferences, improving proficiency in translation \citep{kreutzer-etal-2018-reliability},
summarization \citep{stiennon2022learning,ziegler2020finetuning}, story-telling
\citep{ziegler2020finetuning}, and instruction-following
\citep{ouyang2022training,ramamurthy2023is}. These methods first optimize a neural network reward
function for compatibility with the dataset of preferences under a preference model such as the
Bradley-Terry model \citep{bradley1952rankanalysis}, then fine-tune a language model to maximize the
given reward using reinforcement learning algorithms, commonly REINFORCE
\citep{williams1992reinforce}, proximal policy optimization (PPO; \cite{schulman2017proximal}), or
variants \citep{ramamurthy2023is}. A closely-related line of work leverages LLMs fine-tuned for
instruction following with human feedback to generate additional synthetic preference data for
targeted attributes such as safety or harmlessness \citep{bai2022constitutional}, using only weak
supervision from humans in the form of a text rubric for the LLM's annotations. These methods
represent a convergence of two bodies of work: one body of work on training language models with
reinforcement learning for a variety of
objectives~\citep{Ranzato2015SequenceLT,paulus2018a,wu2018learning} and another body of work on
general methods for learning from human preferences \citep{christiano2017deep,kupcsik2018learning}.
Despite the appeal of using relative human preferences, fine-tuning large language models with
reinforcement learning remains a major practical challenge; this work provides a theoretically-
justified approach to optimizing relative preferences without RL.

Outside of the context of language, learning policies from preferences has been studied in both
bandit and reinforcement learning settings, and several approaches have been proposed. Contextual
bandit learning using preferences or rankings of actions, rather than rewards, is known as a
contextual dueling bandit (CDB; \cite{yue2012karmed,dudik2015contextual}). In the absence of
absolute rewards, theoretical analysis of CDBs substitutes the notion of an optimal policy with a
\textit{von Neumann winner}, a policy whose expected win rate against \textit{any} other policy is
at least 50\% \citep{dudik2015contextual}. However, in the CDB setting, preference labels are given
online, while in learning from human preferences, we typically learn from a fixed batch of offline
preference-annotated action pairs \citep{yan2022human}. Similarly, \textit{preference-based RL}
(PbRL) learns from binary preferences generated by an \textit{unknown} `scoring' function rather
than rewards \citep{BusaFekete2014,ruiz2023dueling}. Various algorithms for PbRL exist, including
methods that can reuse off-policy preference data, but generally involve first explicitly estimating
the latent scoring function (i.e. the reward model) and subsequently optimizing it
\citep{jain2013learning,BusaFekete2014,christiano2017deep,sadigh2017active,kupcsik2018learning}. We
instead present a single stage policy learning approach that directly optimizes a policy to satisfy
preferences.

\section{Preliminaries}\label{section:prelims}

We review the RLHF pipeline in \citeauthor{ziegler2020finetuning} (and later
\citep{stiennon2022learning, bai2022training, ouyang2022training}). It usually includes three
phases: 1) supervised fine-tuning (SFT); 2) preference sampling and reward learning and 3) RL
optimization.

\textbf{SFT}: RLHF typically begins by fine-tuning a pre-trained LM with supervised learning on
high-quality data for the downstream task(s) of interest (dialogue, summarization, etc.), to obtain
a model $\pisft$.

\textbf{Reward Modelling Phase}: In the second phase the SFT model is prompted with prompts $x$ to
produce pairs of answers $(y_1, y_2)\sim \pisft(y \mid x)$. These are then presented to human
labelers who express preferences for one answer, denoted as $y_w\succ y_l \mid x$ where $y_w$ and
$y_l$ denotes the preferred and dispreferred completion amongst $(y_1, y_2)$ respectively. The
preferences are assumed to be generated by some latent reward model $r^*(y, x)$, which we do not
have access to. There are a number of approaches used to model preferences, the Bradley-Terry (BT)
\cite{bradley1952rankanalysis} model being a popular choice (although more general Plackett-Luce
ranking models \citep{plackett1975analysis, luce2012individual} are also compatible with the
framework if we have access to several ranked answers). The BT model stipulates that the human
preference distribution $p^*$ can be written as: \begin{equation}\label{eq:bradley-terry}
p^*(y_1\succ y_2 \mid x)=\frac{\exp\left(r^*(x, y_1)\right)}{\exp\left(r^*(x, y_1)\right) +
\exp\left(r^*(x, y_2)\right)}. \end{equation} Assuming access to a static dataset of comparisons
$\mathcal{D}=\bigl\{x^{(i)}, y_w^{(i)}, y_l^{(i)}\bigr\}_{i=1}^N$ sampled from $p^*$, we can
parametrize a reward model $r_{\phi}(x, y)$ and estimate the parameters via maximum likelihood.
Framing the problem as a binary classification we have the negative log-likelihood loss:
\begin{equation}\label{eq:reward_model}     \mathcal{L}_R(r_{\phi}, \mathcal{D}) = -\mathbb{E}_{(x,
y_w, y_l)\sim \mathcal{D}}\bigl[\log \sigma(r_{\phi}(x, y_w)- r_{\phi}(x, y_l))\bigr] \end{equation}
where $\sigma$ is the logistic function. In the context of LMs, the network $r_{\phi}(x, y)$ is
often initialized from the SFT model $\pisft(y \mid x)$ with the addition of a linear layer on top
of the final transformer layer that produces a single scalar prediction for the reward value
\cite{ziegler2020finetuning}. To ensure a reward function with lower variance, prior works normalize
the rewards, such that  $\mathbb{E}_{x,y\sim \mathcal{D}}\left[r_\phi(x, y)\right] = 0$ for all $x$.

\textbf{RL Fine-Tuning Phase}: During the RL phase, the learned reward function is used to provide
feedback to the language model. Following prior works~\citep{jaques2017sequence, jaques2020human},
the optimization is formulated as \begin{equation}\label{eq:RL} \max_{\pi_{\theta}}
\mathbb{E}_{x\sim \mathcal{D}, y\sim \pi_{\theta}(y \mid x)}\bigl[r_{\phi}(x, y)\bigr] -
\beta\mathbb{D}_{\textrm{KL}}\bigl[\pi_{\theta}(y\mid x)\mid \mid \piref(y\mid x)\bigr],
\end{equation} where $\beta$ is a parameter controlling the deviation from the base reference policy
$\piref$, namely the initial SFT model $\pisft$.  In practice, the language model policy
$\pi_\theta$ is also initialized to $\pisft$. The added constraint is important, as it prevents the
model from deviating too far from the distribution on which the reward model is accurate, as well as
maintaining the generation diversity and preventing mode-collapse to single high-reward answers. Due
to the discrete nature of language generation, this objective is not differentiable and is typically
optimized with reinforcement learning. The standard approach \citep{ziegler2020finetuning,
stiennon2022learning, bai2022training, ouyang2022training} has been to construct the reward function
${r(x, y) = r_{\phi}(x, y) -\beta (\log \pi_{\theta}(y\mid x) - \log \piref(y\mid x))}$, and
maximize using PPO \cite{schulman2017proximal}.

\section{Direct Preference Optimization}\label{sec:DPO}

Motivated by the challenges of applying reinforcement learning algorithms on large-scale problems
such as fine-tuning language models, our goal is to derive a simple approach for policy optimization
using preferences directly. Unlike prior RLHF methods, which learn a reward and then optimize it via
RL, our approach leverages a particular choice of reward model parameterization that enables
extraction of its optimal policy in closed form, without an RL training loop.  As we will describe
next in detail, our key insight is to leverage an analytical mapping from reward functions to
optimal policies, which enables us to transform a loss function over reward functions into a loss
function over policies. This change-of-variables approach avoids fitting an explicit, standalone
reward model, while still optimizing under existing models of human preferences, such as the
Bradley-Terry model. In essence, the policy network represents both the language model and the
(implicit) reward.

\textbf{Deriving the DPO objective.} We start with the same RL objective as prior work,
Eq.~\ref{eq:RL}, under a general reward function $r$. Following prior
work~\citep{peters2007reinforcement, peng2019advantage, korbak2022reinforcement, go2023aligning}, it
is straightforward to show that the optimal solution to the KL-constrained reward maximization
objective in Eq.~\ref{eq:RL} takes the form: \begin{equation}\label{eq:op_policy}     \pi_r(y\mid x)
= \frac{1}{Z(x)}\piref(y\mid x)\exp\left(\frac{1}{\beta}r(x, y)\right), \end{equation}% where $Z(x)
=\sum_{y}\piref(y\mid x)\exp\left(\frac{1}{\beta}r(x, y)\right)$ is the partition function. See
Appendix \ref{app:derivation1} for a complete derivation. Even if we use the MLE estimate $r_{\phi}$
of the ground-truth reward function $r^*$, it is still expensive to estimate the partition function
$Z(x)$ \citep{korbak2022reinforcement, go2023aligning}, which makes this representation hard to
utilize in practice. However, we can rearrange Eq.~\ref{eq:op_policy} to express the reward function
in terms of its corresponding optimal policy $\pi_r$, the reference policy $\piref$, and the unknown
partition function $Z(\cdot)$. Specifically, we first take the logarithm of both sides of
Eq.~\ref{eq:op_policy} and then with some algebra we obtain: \begin{equation}\label{eq:main_eq}
r(x,y) =\beta \log \frac{\pi_r(y\mid x)}{\piref(y\mid x)} + \beta \log Z(x). \end{equation} We can
apply this reparameterization to the ground-truth reward $r^*$ and corresponding optimal model
$\pi^*$. Fortunately, the Bradley-Terry model depends only on the difference of rewards between two
completions, i.e., ${p^*(y_1 \succ y_2 \mid x) = \sigma(r^*(x, y_1) - r^*(x, y_2))}$. Substituting
the reparameterization in Eq.~\ref{eq:main_eq} for $r^*(x,y)$ into the preference model
Eq.~\ref{eq:bradley-terry}, the partition function cancels, and we can express the human preference
probability in terms of only the optimal policy $\pi^*$ and reference policy $\piref$. Thus, the
optimal RLHF policy $\pi^*$ under the Bradley-Terry model satisfies the preference model:
\begin{equation}\label{eq:objective}     p^*(y_1\succ y_2 \mid x)=\frac{1}{1 + \exp\left(\beta \log
\frac{\pi^*(y_2\mid x)}{\piref(y_2\mid x)} - \beta \log \frac{\pi^*(y_1\mid x)}{\piref(y_1\mid
x)}\right)} \end{equation} The derivation is in Appendix~\ref{app:derivation2}. While
Eq.~\ref{eq:objective} uses the Bradley-Terry model, we can similarly derive expressions under the
more general Plackett-Luce models~\citep{plackett1975analysis, luce2012individual}, shown in
Appendix~\ref{app:plackett_luce_models}.

Now that we have the probability of human preference data in terms of the optimal policy rather than
the reward model, we can formulate a maximum likelihood objective for a parametrized policy
$\pi_\theta$. Analogous to the reward modeling approach (i.e. Eq.~\ref{eq:reward_model}), our policy
objective becomes: \begin{equation}\label{eq:optimum_model}     \mathcal{L}_\text{DPO}(\pi_{\theta};
\piref) = -\mathbb{E}_{(x, y_w, y_l)\sim \mathcal{D}}\left[\log \sigma \left(\beta \log
\frac{\pi_{\theta}(y_w\mid x)}{\piref(y_w\mid x)} - \beta \log \frac{\pi_{\theta}(y_l\mid
x)}{\piref(y_l\mid x)}\right)\right]. \end{equation} This way, we fit an implicit reward using an
alternative parameterization, whose optimal policy is simply $\pi_\theta$. Moreover, since our
procedure is equivalent to fitting a reparametrized Bradley-Terry model, it enjoys certain
theoretical properties, such as consistencies under suitable assumption of the preference data
distribution \cite{bong2022generalized}. In Section~\ref{sec:theory}, we further discuss theoretical
properties of DPO in relation to other works.

\textbf{What does the DPO update do?} For a mechanistic understanding of DPO, it is useful to
analyze the gradient of the loss function $\mathcal{L}_\text{DPO}$. The gradient with respect to the
parameters $\theta$ can be written as: \begin{multline*}\label{eq:gradient}     \nabla_\theta
\mathcal{L}_\text{DPO}(\pi_\theta;\piref) = \\ -\beta\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}}
\bigg[\underbrace{\sigma(\hat{r}_\theta(x, y_l) - \hat{r}_\theta (x, y_w))}_\text{higher weight when
reward estimate is wrong}\bigg[\underbrace{\nabla_\theta\log \pi(y_w \mid x)}_\text{increase
likelihood of $y_w$} - \underbrace{\nabla_\theta\log\pi(y_l \mid x)}_\text{decrease likelihood of
$y_l$}\bigg]\bigg], \end{multline*} where $\hat{r}_\theta(x, y) = \beta \log \frac{\pi_\theta(y \mid
x)}{\piref(y \mid x)}$ is the reward implicitly defined by the language model $\pi_\theta$ and
reference model $\piref$ (more in Section~\ref{sec:theory}). Intuitively, the gradient of the loss
function $\mathcal{L}_\text{DPO}$ increases the likelihood of the preferred completions $y_w$ and
decreases the likelihood of dispreferred completions $y_l$. Importantly, the examples are weighed by
how much higher the implicit reward model $\hat{r}_\theta$ rates the dispreferred completions,
scaled by $\beta$, i.e, how incorrectly the implicit reward model orders the completions, accounting
for the strength of the KL constraint. Our experiments suggest the importance of this weighting, as
a na\"ive version of this method without the weighting coefficient can cause the language model to
degenerate (Appendix Table~\ref{tab:unlikelihood_generations}).

\textbf{DPO outline.}  The general DPO pipeline is as follows: 1) Sample completions $y_1, y_2 \sim
\piref(\cdot \mid x)$ for every prompt $x$, label with human preferences to construct the offline
dataset of preferences $\mathcal{D} = \{x^{(i)}, y_w^{(i)}, y_l)^{(i)}\}_{i=1}^N$ and 2) optimize
the language model $\pi_\theta$ to minimize $\mathcal{L}_\text{DPO}$ for the given $\piref$ and
$\mathcal{D}$ and desired $\beta$.  In practice, one would like to reuse preference datasets
publicly available, rather than generating samples and gathering human preferences. Since the
preference datasets are sampled using $\pisft$, we initialize $\piref = \pisft$ whenever available.
However, when $\pisft$ is not available, we initialize $\piref$ by maximizing likelihood of
preferred completions ${(x, y_w)}$, that is, ${\piref = \argmax_{\pi}\mathbb{E}_{x, y_w \sim
\mathcal{D}}\left[\log \pi(y_w \mid x)\right]}$. This procedure helps mitigate the distribution
shift between the true reference distribution which is unavailable, and $\piref$ used by DPO.
Further details related to the implementation and hyperparameters can be found in
Appendix~\ref{app:implementation}.

\section{Theoretical Analysis of DPO} In this section, we give further interpretation of the DPO
method, provide theoretical backing, and relate advantages of DPO to issues with actor critic
algorithms used for RLHF (such as PPO~\cite{schulman2017proximal}).

\label{sec:theory}

\subsection{Your Language Model Is Secretly a Reward Model} DPO is able to bypass both fitting an
explicit reward and performing RL to learn the policy using a single maximum likelihood objective.
Note the optimization objective Eq. \ref{eq:main_eq} is equivalent to a Bradley-Terry model with a
reward parameterization $r^*(x, y) = \beta \log\frac{\pi^*_\theta(y \mid x)}{\piref(y \mid x)}$ and
we optimize our parametric model $\pi_{\theta}$, equivalently to the reward model optimization in
Eq. \ref{eq:reward_model} under the change of variables. In this section we will build the theory
behind this reparameterization, show that it does not constrain the class of learned reward models,
and allows for the exact recovery of the optimal policy. We begin with by defining an equivalence
relation between reward functions.

\begin{definition} We say that two reward functions $r(x, y)$ and $r'(x, y)$ are equivalent iff
${r(x, y)-r'(x, y) = f(x)}$ for some function $f$.      \end{definition} It is easy to see that this
is indeed an equivalence relation, which partitions the set of reward functions into classes. We can
state the following two lemmas:

\begin{lemma}\label{lemma:same_prefrence} Under the Plackett-Luce, and in particular the Bradley-
Terry, preference framework, two reward functions from the same class induce the same preference
distribution. \end{lemma}

\begin{lemma}\label{lemma:same_policy}     Two reward functions from the same equivalence class
induce the same optimal policy under the constrained RL problem. \end{lemma} The proofs are
straightforward and we defer them to Appendix \ref{app:lemma1}. The first lemma is a well-known
under-specification issue with the Plackett-Luce family of models \cite{plackett1975analysis}. Due
to this under-specification, we usually have to impose additional identifiability constraints to
achieve any guarantees on the MLE estimates from Eq. \ref{eq:reward_model}
\cite{bong2022generalized}. The second lemma states that all reward functions from the same class
yield the same optimal policy, hence for our final objective, we are only interested in recovering
an arbitrary reward function from the optimal class. We prove the following Theorem in
Appendix~\ref{app:thm1}: \begin{theorem}\label{thm:main}     Under mild assumptions, all reward
classes consistent with the Plackett-Luce (and Bradley-Terry in particular) models can be
represented with the reparameterization ${r(x, y) = \beta \log \frac{\pi(y\mid x)}{\piref(y\mid
x)}}$ for some model $\pi(y\mid x)$ and a given reference model $\piref(y \mid x)$. \end{theorem}
\begin{sproof}     Consider any reward function $r(x, y)$, which induces a corresponding optimal
model $\pi_r(y \mid x)$, specified by Eq. \ref{eq:op_policy}. We will show that a reward function
from the equivalence class of $r$ can be represented using the reparameterization given above. We
define the projection $f$ as   \begin{equation}     f(r; \piref, \beta)(x, y) = r(x, y) -
\beta\log\sum_{y}\piref(y\mid x)\exp\left(\frac{1}{\beta}r(x, y)\right) \end{equation} The operator
$f$ simply normalizes the reward function with the logarithm of the partition function of $\pi_r$.
Since the added normalization term is only a function of the prefix $x$, $f(r; \piref, \beta)(x, y)
$ is a reward function in the equivalence class of $r(x, y)$. Finally, replacing $r$ with the RHS of
Eq.~\ref{eq:main_eq} (which holds for any reward function), we have $f(r; \piref, \beta)(x, y) =
\beta \log \frac{\pi_r(y\mid x)}{\piref(y\mid x)}$. That is, the projection $f$ produces a member of
the equivalence class of $r$ with the desired form, and we do not lose any generality in our reward
model from the proposed reparameterization. \end{sproof} We can alternatively view
Theorem~\ref{thm:main} as specifying exactly which reward function within each equivalence class the
DPO reparameterization selects, that is, the reward function satisfying:
\begin{equation}\label{eq:lag_p}      \sum_{y}\underbrace{\piref(y\mid
x)\exp\left(\frac{1}{\beta}r(x, y)\right)}_{=\pi(y\mid x)\text{, using Thm.~\ref{thm:main}
reparam.}} = 1, \end{equation} i.e., $\pi(y\mid x)$ is a valid distribution (probabilities are
positive and sum to 1). However, following Eq.~\ref{eq:op_policy}, we can see that
Eq.~\ref{eq:lag_p} is the partition function of the optimal policy induced by the reward function
$r(x, y)$. The key insight of the DPO algorithm is that we can impose certain constraints on the
under-constrained Plackett-Luce (and Bradley-Terry in particular) family of preference models, such
that we preserve the class of representable reward models, but explicitly make the optimal policy in
Eq. \ref{eq:op_policy} analytically tractable for all prompts $x$.

\subsection{Instability of Actor-Critic Algorithms} We can also use our framework to diagnose
instabilities with standard actor-critic algorithms used for the RLHF, such as PPO. We follow the
RLHF pipeline and focus on the RL fine-tuning step outlined in Section \ref{section:prelims}. We can
draw connections to the control as inference framework \cite{levine2018reinforcement} for the
constrained RL problem outlined in \ref{eq:RL}. We assume a parameterized model $\pi_{\theta}(y\mid
x)$ and minimize $\mathbb{D}_{\text{KL}}[\pi_{\theta}(y|x) \mid \mid \pi^*(y\mid x)]$ where $\pi^*$
is the optimal policy from Eq. \ref{eq:optimum_model} induced by the reward function $r_{\phi}(y,
x)$. With some algebra this leads to the optimization objective: \begin{equation}\label{eq:AC}
\max_{\pi_{\theta}}\mathbb{E}_{\pi_{\theta}(y\mid x)}\bigg[\underbrace{r_{\phi}(x, y)
-\beta\log\sum_{y}\piref(y\mid x)\exp\left(\frac{1}{\beta}r_{\phi}(x, y)\right)}_{f(r_{\phi},
\piref, \beta)} - \underbrace{\beta\log\frac{\pi_{\theta}(y\mid x)}{\piref(y\mid
x)}}_{\text{KL}}\bigg] \end{equation} This is the same objective optimized in prior works
\citep{ziegler2020finetuning, stiennon2022learning, bai2022training, ouyang2022training} using the
DPO-equivalent reward for the reward class of $r_{\phi}$. In this setting, we can interpret the
normalization term in $f(r_{\phi}, \piref, \beta)$ as the soft value function of the reference
policy $\piref$. While this term does not affect the optimal solution, without it, the policy
gradient of the objective could have high variance, making learning unstable. We can accommodate for
the normalization term using a learned value function, but that can also be difficult to optimize.
Alternatively, prior works have normalized rewards using a human completion baseline, essentially a
single sample Monte-Carlo estimate of the normalizing term. In contrast the DPO reparameterization
yields a reward function that does not require any baselines.

\section{Experiments} In this section, we empirically evaluate DPO's ability to train policies
directly from preferences. First, in a well-controlled text-generation setting, we ask: how
efficiently does DPO trade off maximizing reward and minimizing KL-divergence with the reference
policy, compared to common preference learning algorithms such as PPO? Next, we evaluate DPO's
performance on larger models and more difficult RLHF tasks, including summarization and dialogue. We
find that with almost no tuning of hyperparameters, DPO tends to perform as well or better than
strong baselines like RLHF with PPO as well as returning the best of $N$ sampled trajectories under
a learned reward function. Before presenting these results, we describe the experimental set-up;
additional details are in Appendix~\ref{app:exp_details}.

\textbf{Tasks.} Our experiments explore three different open-ended text generation tasks. For all
experiments, algorithms learn a policy from a dataset of preferences $\mathcal{D}=\bigl\{x^{(i)},
y_w^{(i)}, y_l^{(i)}\bigr\}_{i=1}^N$. In \textbf{controlled sentiment generation}, $x$ is a prefix
of a movie review from the IMDb dataset \cite{maas-EtAl:2011:ACL-HLT2011}, and the policy must
generate $y$ with positive sentiment. In order to perform a controlled evaluation, for this
experiment we \textit{generate} preference pairs over generations using a pre-trained sentiment
classifier, where $p(\text{positive}\mid x,y_w)>p(\text{positive}\mid x,y_l)$. For SFT, we fine-tune
GPT-2-large until convergence on reviews from the train split of the IMDB dataset (further details
in App~\ref{app:sentiment_details}). In \textbf{summarization}, $x$ is a forum post from Reddit; the
policy must generate a summary $y$ of the main points in the post. Following prior work, we use the
Reddit TL;DR summarization dataset \citep{volske-etal-2017-tl} along with human preferences gathered
by \citeauthor{stiennon2022learning}. We use an SFT model fine-tuned on human-written forum post
summaries\footnote{\url{https://huggingface.co/CarperAI/openai_summarize_tldr_sft}} with the TRLX
\citep{leandro_von_werra_2023_7790115} framework for RLHF. The human preference dataset was gathered
by \citeauthor{stiennon2022learning} on samples from a different, but similarly-trained, SFT model.
Finally, in \textbf{single-turn dialogue},  $x$ is a human query, which may be anything from a
question about astrophysics to a request for relationship advice. A policy must produce an engaging
and helpful response $y$ to a user's query; we use the Anthropic Helpful and Harmless dialogue
dataset \citep{bai2022training}, containing 170k dialogues between a human and an automated
assistant. Each transcript ends with a pair of responses generated by a large (although unknown)
language model along with a preference label denoting the human-preferred response. In this setting,
no pre-trained SFT model is available; we therefore fine-tune an off-the-shelf language model on
only the preferred completions to form the SFT model.

\begin{figure}     \centering
\includegraphics[width=0.50\textwidth]{figures/results/frontier.pdf}
\includegraphics[width=0.49\textwidth]{figures/results/tldr_winrate_vs_temp.pdf}
\caption{\textbf{Left.} The frontier of expected reward vs KL to the reference policy. DPO provides
the highest expected reward for all KL values, demonstrating the quality of the optimization.
\textbf{Right.} TL;DR summarization win rates vs. human-written summaries, using GPT-4 as evaluator.
DPO exceeds PPO's best-case performance on summarization, while being more robust to changes in the
sampling temperature.}     \vspace{-2mm}     \label{fig:frontier-tldr-main} \end{figure}

\textbf{Evaluation.} Our experiments use two different approaches to evaluation. In order to analyze
the effectiveness of each algorithm in optimizing the constrained reward maximization objective, in
the controlled sentiment generation setting we evaluate each algorithm by its frontier of achieved
reward and KL-divergence from the reference policy; this frontier is computable because we have
acccess to the ground-truth reward function (a sentiment classifier). However, in the real world,
the ground truth reward function is not known; therefore, we evaluate algorithms with their
\textit{win rate} against a baseline policy, using GPT-4 as a proxy for human evaluation of summary
quality and response helpfulness in the summarization and single-turn dialogue settings,
respectively. For summarization, we use reference summaries in the test set as the baseline; for
dialogue, we use the preferred response in the test dataset as the baseline. While existing studies
suggest LMs can be better automated evaluators than existing metrics \citep{Chen2023ExploringTU}, we
conduct a human study to justify our usage of GPT-4 for evaluation in Sec.~\ref{sec:human-
judgments}. We find GPT-4 judgments correlate strongly with humans, with human agreement with GPT-4
typically similar or higher than inter-human annotator agreement.

\textbf{Methods.} In addition to DPO, we evaluate several existing approaches to training language
models to adhere to human preferences. Most simply, we explore zero-shot prompting with
\textbf{GPT-J} \citep{gpt-j} in the summarization task and 2-shot prompting with
\textbf{Pythia-2.8B} \citep{biderman2023pythia} in the dialogue task. In addition, we evaluate the
\textbf{SFT} model as well as \textbf{Preferred-FT}, which is a model fine-tuned with supervised
learning on the chosen completion $y_w$ from either the SFT model (in controlled sentiment and
summarization) or a generic LM (in single-turn dialogue). Another pseudo-supervised method is
\textbf{Unlikelihood}~\citep{welleck2019neural}, which simply optimizes the policy to maximize the
probability assigned to $y_w$ and \textit{minimize} the probability assigned to $y_l$; we use an
optional coefficient $\alpha\in[0,1]$ on the `unlikelihood' term. We also consider \textbf{PPO}
\citep{schulman2017proximal} using a reward function learned from the preference data and
\textbf{PPO-GT}, which is an oracle that learns from the ground truth reward function available in
the controlled sentiment setting. In our sentiment experiments, we use two implementations of PPO-
GT, one of-the-shelf version \cite{leandro_von_werra_2023_7790115} as well as a modified version
that normalizes rewards and further tunes hyperparameters to improve performance (we also use these
modifications when running `normal' PPO with learned rewards). Finally, we consider the \textbf{Best
of $N$} baseline, sampling $N$ responses from the SFT model (or Preferred-FT in dialogue) and
returning the highest-scoring response according to a reward function learned from the preference
dataset. This high-performing method decouples the quality of the reward model from the PPO
optimization, but is computationally impractical even for moderate $N$ as it requires sampling $N$
completions for every query at test time.

\subsection{How well can DPO optimize the RLHF objective?}

\begin{figure}     \centering
\includegraphics[width=0.50\textwidth]{figures/results/dialogue_winrate_vs_temp.pdf}
\includegraphics[width=0.49\textwidth]{figures/results/dialogue_winrate_vs_steps.pdf}
\caption{\textbf{Left.} Win rates computed by GPT-4 for Anthropic-HH one-step dialogue; DPO is the
only method that improves over chosen summaries in the Anthropic-HH test set. \textbf{Right.} Win
rates for different sampling temperatures over the course of training. DPO's improvement over the
dataset labels is fairly stable over the course of training for different sampling temperatures.}
\vspace{-2mm}     \label{fig:dialogue-main} \end{figure}

The KL-constrained reward maximization objective used in typical RLHF algorithms balances
exploitation of reward while restricting the policy from deviating far from the reference policy.
Therefore, when comparing algorithms, we must take into account both reward achieved as well as the
KL discrepancy; achieving slightly higher reward but with much higher KL is not necessarily
desirable. Figure~\ref{fig:frontier-tldr-main} shows the reward-KL frontier for various algorithms
in the sentiment setting. We execute multiple training runs for each algorithm, using a different
hyperparameter for policy conservativeness in each run (target KL $\in\{3,6,9,12\}$ for PPO, $\beta
\in \{0.05,0.1,1,5\}$, $\alpha\in\{0.05,0.1,0.5,1\}$ for unlikelihood, random seeds for preferred-
FT). This sweep includes 22 runs in total. After each 100 training steps until convergence, we
evaluate each policy on a set of test prompts, computing the average reward under the true reward
function as well as the average sequence-level KL\footnote{That is, the sum of the per-timestep KL-
divergences.} with the reference policy $\text{KL}\left(\pi\mid \mid \piref\right)$. We find that
DPO produces by far the most efficient frontier, achieving the highest reward while still achieving
low KL. This result is particularly notable for multiple reasons. First, DPO and PPO optimize the
same objective, but DPO is notably more efficient; DPO's reward/KL tradeoff strictly dominates PPO.
Second, DPO achieves a better frontier than PPO, \emph{even when PPO can access ground truth
rewards} (PPO-GT).

\subsection{Can DPO scale to real preference datasets?} \label{sec:dpo-real-datasets} Next, we
evaluate fine-tuning performance of DPO on summarization and single-turn dialogue. For
summarization, automatic evaluation metrics such as ROUGE can be poorly correlated with human
preferences~\citep{stiennon2022learning}, and prior work has found that fine-tuning LMs using PPO on
human preferences to provide more effective summaries. We evaluate different methods by sampling
completions on the test split of TL;DR summarization dataset, and computing the average win rate
against reference completions in the test set. The completions for all methods are sampled at
temperatures varying from 0.0 to 1.0, and the win rates are shown in Figure~\ref{fig:frontier-tldr-
main} (right). DPO, PPO and Preferred-FT all fine-tune the same GPT-J SFT
model\footnote{\url{https://huggingface.co/CarperAI/openai_summarize_tldr_sft}}. We find that DPO
has a win rate of approximately 61\% at a temperature of 0.0, exceeding the performance of PPO at
~57\% at its optimal sampling temperature of 0.0. DPO also achieves a higher maximum win rate
compared to the best of $N$ baseline. We note that we did not meaningfully tune DPO's $\beta$
hyperparameter, so these results may underestimate DPO's potential. Moreover, we find DPO to be much
more robust to the sampling temperature than PPO, the performance of which can degrade to that of
the base GPT-J model at high temperatures. Preferred-FT does not improve significantly over the SFT
model. We also compare DPO and PPO head-to-head in human evaluations in Section~\ref{sec:human-
judgments}, where DPO samples at temperature 0.25 were preferred 58\% times over PPO samples at
temperature 0.

On single-turn dialogue, we evaluate the different methods on the subset of the test split of the
Anthropic HH dataset \citep{bai2022training} with one step of human-assistant interaction. GPT-4
evaluations use the preferred completions on the test as the reference to compute the win rate for
different methods. As there is no standard SFT model for this task, we start with a pre-trained
Pythia-2.8B, use Preferred-FT to train a reference model on the chosen completions such that
completions are within distribution of the model, and then train using DPO. We also compare against
the best of 128 Preferred-FT completions (we found the Best of $N$ baseline plateaus at 128
completions for this task; see Appendix Figure~\ref{fig:best-of-n}) and a 2-shot prompted version of
the Pythia-2.8B base model, finding DPO performs as well or better for the best-performing
temperatures for each method. We also evaluate an RLHF model trained with PPO on the Anthropic HH
dataset \footnote{\url{https://huggingface.co/reciprocate/ppo_hh_pythia-6B}} from a well-known
source \footnote{\url{https://github.com/CarperAI/trlx/tree/main/examples/hh}}, but are unable to
find a prompt or sampling temperature that gives performance better than the base Pythia-2.8B model.
Based on our results from TL;DR and the fact that both methods optimize the same reward function, we
consider Best of 128 a rough proxy for PPO-level performance. Overall, DPO is the only
computationally efficient method that improves over the preferred completions in the Anthropic HH
dataset, and provides similar or better performance to the computationally demanding Best of 128
baseline. Finally, Figure~\ref{fig:dialogue-main} shows that DPO converges to its best performance
relatively quickly.

\subsection{Generalization to a new input distribution}

\begin{wraptable}{r}{0.375\textwidth}     \small     \vspace{-10mm}     \begin{tabular}{ccc}
\toprule         & \multicolumn{2}{c}{\textbf{Win rate vs. ground truth}} \\
\cmidrule(lr){2-3}         \textbf{Alg.} & Temp $0$ & Temp $0.25$ \\         \midrule         DPO &
0.36 & 0.31 \\         PPO & 0.26 & 0.23 \\         \bottomrule     \end{tabular}     \caption{GPT-4
win rates vs. ground truth summaries for out-of-distribution CNN/DailyMail input articles.}
\vspace{-3mm}     \label{tab:ood} \end{wraptable}

To further compare the performance of PPO and DPO under distribution shifts, we evaluate the PPO and
DPO policies from our Reddit TL;DR summarization experiment on a different distribution, news
articles in the test split of the CNN/DailyMail dataset \citep{nallapati-etal-2016-abstractive},
using the best sampling temperatures from TL;DR (0 and 0.25). The results are presented in
Table~\ref{tab:ood}. We computed the GPT-4 win rate against the ground-truth summaries in the
datasets, using the same GPT-4 (C) prompt we used for Reddit TL;DR, but replacing the words ``forum
post'' with ``news article''. For this new distribution, DPO continues to outperform the PPO policy
by a significant margin. This experiment provides initial evidence that DPO policies can generalize
similarly well to PPO policies, even though DPO does not use the additional unlabeled Reddit TL;DR
prompts that PPO uses.

\subsection{Validating GPT-4 judgments with human judgments} \label{sec:human-judgments} We conduct
a human study to verify the reliability of GPT-4's judgments, using the results of the TL;DR
summarization experiment and two different GPT-4 prompts. The \textbf{GPT-4 (S)} (simple) prompt
simply asks for which summary better-summarizes the important information in the post. The
\textbf{GPT-4 (C)} (concise) prompt also asks for which summary is more concise; we evaluate this
prompt because we find that GPT-4 prefers longer, more repetitive summaries than humans do with the
\textbf{GPT-4 (S)} prompt. See Appendix~\ref{app:prompts} for the complete prompts. We perform three
comparisons, using the highest (DPO, temp. 0.25), the lowest (PPO, temp. 1.0), and a
\begin{wraptable}{r}{0.47\textwidth}     \centering     \small     \vspace{-1.5mm}
\begin{tabular}{lccc}     \toprule         & \textbf{DPO} & \textbf{SFT} & \textbf{PPO-1} \\
\cmidrule(lr){2-4}         N respondents & 272 & 122 & 199 \\         \midrule         GPT-4 (S) win
\% & 47 & 27 & 13 \\         GPT-4 (C) win \% & 54 & 32 & 12 \\         Human win \% & 58 & 43 & 17
\\         \midrule         GPT-4 (S)-H agree & 70 & 77 & 86 \\         GPT-4 (C)-H agree & 67 & 79
& 85 \\         H-H agree & 65 & - & 87 \\         \bottomrule     \end{tabular}     \vspace{-1mm}
\caption{Comparing human and GPT-4 win rates and per-judgment agreement on TL;DR summarization
samples. \textbf{Humans agree with GPT-4 about as much as they agree with each other.} Each
experiment compares a summary from the stated method with a summary from PPO with temperature 0.}
\vspace{-5mm}     \label{tab:human_results} \end{wraptable}middle-performing (SFT, temp. 0.25)
method with the aim of covering a diversity of sample qualities; all three methods are compared
against greedily-sampled PPO (its best-performing temperature). We find that with both prompts,
GPT-4 tends to agree with humans about as often as humans agree with each other, suggesting that
GPT-4 is a reasonable proxy for human evaluations (due to limited human raters, we only collect
multiple human judgments for the DPO and PPO-1 comparisons). Overall, the \textbf{GPT-4 (C)} prompt
generally provides win rates more representative of humans; we therefore use this prompt for the
main results in Section~\ref{sec:dpo-real-datasets}. For additional details about the human study,
including the web interface presented to raters and the list of human volunteers, see
Appendix~\ref{app:human-study}.

\section{Discussion} Learning from preferences is a powerful, scalable framework for training
capable, aligned language models. We have introduced DPO, a simple training paradigm for training
language models from preferences without reinforcement learning. Rather than coercing the preference
learning problem into a standard RL setting in order to use off-the-shelf RL algorithms, DPO
identifies a mapping between language model policies and reward functions that enables training a
language model to satisfy human preferences \textit{directly}, with a simple cross-entropy loss,
without reinforcement learning or loss of generality. With virtually no tuning of hyperparameters,
DPO performs similarly or better than existing RLHF algorithms, including those based on PPO; DPO
thus meaningfully reduces the barrier to training more language models from human preferences.

\textbf{Limitations \& Future Work.} Our results raise several important questions for future work.
How does the DPO policy generalize out of distribution, compared with learning from an explicit
reward function? Our initial results suggest that DPO policies can generalize similarly to PPO-based
models, but more comprehensive study is needed. For example, can training with self-labeling from
the DPO policy similarly make effective use of unlabeled prompts? On another front, how does reward
over-optimization manifest in the direct preference optimization setting, and is the slight decrease
in performance in Figure~\ref{fig:dialogue-main}-right an instance of it? Additionally, while we
evaluate models up to 6B parameters, exploration of scaling DPO to state-of-the-art models orders of
magnitude larger is an exciting direction for future work. Regarding evaluations, we find that the
win rates computed by GPT-4 are impacted by the prompt; future work may study the best way to elicit
high-quality judgments from automated systems. Finally, many possible applications of DPO exist
beyond training language models from human preferences, including training generative models in
other modalities.

\end{document}

In [2]:
import re
import pandas as pd

def parse_paper_structure(text):
    """Parse paper into sections, subsections and paragraphs with metadata."""
    sections = []
    
    # Split by sections first
    section_pattern = r'\\section\{([^}]+)\}'
    section_splits = re.split(section_pattern, text)
    
    current_section = "Title/Abstract"
    current_section_content = ""
    
    for i in range(len(section_splits)):
        if i == 0:
            # Content before first section
            content = section_splits[i]
            current_section_content = content
        elif i % 2 == 1:
            # This is a section title
            current_section = section_splits[i]
            continue
        else:
            # This is section content
            content = section_splits[i]
            current_section_content = content
        
        # Now split by subsections within this section
        subsection_pattern = r'\\subsection\{([^}]+)\}'
        subsection_splits = re.split(subsection_pattern, content)
        
        current_subsection = "No Subsection"
        current_subsection_content = ""
        
        for j in range(len(subsection_splits)):
            if j == 0:
                # Content before first subsection
                subsection_content = subsection_splits[j]
                current_subsection_content = subsection_content
            elif j % 2 == 1:
                # This is a subsection title
                current_subsection = subsection_splits[j]
                continue
            else:
                # This is subsection content
                subsection_content = subsection_splits[j]
                current_subsection_content = subsection_content
            
            # Split into paragraphs
            paragraphs = [p.strip() for p in subsection_content.split('\n\n') if p.strip()]
            
            for paragraph in paragraphs:
                sections.append({
                    'section': current_section,
                    'subsection': current_subsection,
                    'paragraph': paragraph,
                    'section_text': current_section_content,
                    'subsection_text': current_subsection_content
                })
    
    return pd.DataFrame(sections)

### 1. Extracting Knowledge

This is to tag sentences that contain information, knowledge, and facts. From these we will extract self-contained, atomic facts.

In [3]:
extraction_prompt = r"""Your task is to act as a text segmenter. Carefully read the provided text from a paper and identify all sentences that contain "pieces of knowledge" or "facts."

# What to Exclude (Do NOT tag these):
- Sentences that are transitional and for structural purposes of the paper, mainly containing language that's generic to any paper, adding zero information e.g. "Our results raise several important questions for future work.", "In this section, we discuss our methodology in relation to other works.
- Author Speculation or Rhetorical Questions: Subjective statements or questions posed to the reader (e.g., "This result is quite surprising.", "But what if the model could...?"). 
- Figures and Tables: Latex commands that generate figures and tables.

# Instructions
1.  Read the entire text carefully.
2.  Identify all sentences that contain a "piece of knowledge" or a "fact" and do not fall into the exclusion categories.
3.  Wrap each of these sentences in `<knowledge>` and `</knowledge>` tags.
4.  You can tag *captions* of the table or figure, but please DO NOT tag the other parts of the table/figure.
5.  For sentences that contain latex commands, place the tags so that it includes any latex code that's part of the sentence e.g. "\begin{definition}" or "\text{...}".
6.  For sentences that contain math, make sure to include all the latex of the math within the tags.
7.  Please make sure the tags cover the ENTIRE sentence i.e. the tags are at the beginning and end of the sentence.
8.  Return the entire original text with these annotations. Do not modify or summarize the text itself."""

# Parse paper structure
paper_df = parse_paper_structure(paper)

# Process each paragraph with LLM
import concurrent.futures

def query_single(paragraph):
    prompt = {}
    prompt['system'] = extraction_prompt
    prompt['user'] = f"""{paragraph}"""
    return utils.query_llm(prompt, model='gpt-4.1')

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(query_single, row['paragraph']) for _, row in paper_df.iterrows()]
    extracted_claims = [future.result() for future in futures]

# Add extracted claims to dataframe
paper_df['extracted_claims'] = extracted_claims

# Print each output with section/subsection context
for i, (_, row) in enumerate(paper_df.iterrows(), 1):
    print_wrapped(f"Section: {row['section']}")
    print_wrapped(f"Subsection: {row['subsection']}")
    print_wrapped(f"Paragraph {i}: {row['extracted_claims']}")
    print_wrapped("-" * 50)

Section: Title/Abstract

Subsection: No Subsection

Paragraph 1: \title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}

--------------------------------------------------

Section: Title/Abstract

Subsection: No Subsection

Paragraph 2: \begin{abstract} <knowledge>While large-scale unsupervised language models (LMs) learn
broad world knowledge and some reasoning skills, achieving precise control of their behavior is
difficult due to the completely unsupervised nature of their training.</knowledge>
<knowledge>Existing methods for gaining such steerability collect human labels of the relative
quality of model generations and fine-tune the unsupervised LM to align with these preferences,
often with reinforcement learning from human feedback (RLHF).</knowledge> <knowledge>However, RLHF
is a complex and often unstable procedure, first fitting a reward model that reflects the human
preferences, and then fine-tuning the large unsupervised LM using reinforcemen

#### 1.1 Check Knowledge is actually in the paper

In [4]:
import re
import pandas as pd

def extract_text_from_knowledge_tags(text: str) -> list[str]:
    """
    Finds all <knowledge> tags in a given text and extracts their content.

    Args:
        text: A string containing the text to parse, which may include
              <knowledge>...</knowledge> tags.

    Returns:
        A list of strings, where each string is the content found within
        a <knowledge> tag. The content is stripped of leading/trailing
        whitespace.
    """
    # This regex pattern finds all content between <knowledge> and </knowledge>.
    # - The (.*?) part is a non-greedy capture group for the content inside the tags.
    # - The re.DOTALL flag allows the '.' character to match newlines, so tags
    #   that span multiple lines are correctly handled.
    pattern = re.compile(r'<knowledge>(.*?)</knowledge>', re.DOTALL)
    
    # re.findall returns a list of all captured groups.
    matches = pattern.findall(text)
    
    # Clean up any leading/trailing whitespace from the extracted text.
    cleaned_matches = [match.strip() for match in matches]
    
    return cleaned_matches

def remove_knowledge_tags(text: str) -> str:
    """Remove knowledge tags from text while preserving the content."""
    pattern = re.compile(r'</?knowledge>', re.DOTALL)
    return pattern.sub('', text)

paper_df['extracted_claims'] = extracted_claims

# Extract a list of knowledge statements for each row
paper_df['knowledge_list'] = paper_df['extracted_claims'].apply(extract_text_from_knowledge_tags)

# Explode the DataFrame on the knowledge_list column
paper_df_exploded = paper_df.explode('knowledge_list').rename(columns={'knowledge_list': 'raw_knowledge_statement'})

# Drop rows with no knowledge statements
paper_df_exploded = paper_df_exploded[paper_df_exploded['raw_knowledge_statement'].notna()]

# Count total claims extracted by the LLM before filtering
total_extracted_claims = paper_df_exploded['raw_knowledge_statement'].notna().sum()

# Filter out claims that are not actually in the original paper text (case-insensitive)
paper_lower = paper.lower()
def is_claim_in_paper(claim):
    # Rows with no knowledge statement (claim is NaN) are kept
    if pd.isna(claim):
        return True
    # Check if the lowercased claim is in the lowercased paper
    return claim.strip().lower() in paper_lower

# Apply the filter and create a new validated dataframe
paper_df_validated = paper_df_exploded[paper_df_exploded['raw_knowledge_statement'].apply(is_claim_in_paper)].copy()

# Count claims that passed validation
validated_claims_count = paper_df_validated['raw_knowledge_statement'].notna().sum()
print(f"Found {validated_claims_count}/{total_extracted_claims} extracted claims in the original paper text.")

# Clean up the original paragraph text by removing knowledge tags from the validated dataframe
paper_df_validated['paragraph'] = paper_df_validated['extracted_claims'].apply(remove_knowledge_tags)

# Drop the now-redundant columns
paper_df_validated = paper_df_validated.drop(columns=['extracted_claims'])

# Display the result
print(f"Total rows after exploding and validation: {len(paper_df_validated)}")
paper_df_validated

Found 200/200 extracted claims in the original paper text.
Total rows after exploding and validation: 200


,section,subsection,paragraph,section_text,subsection_text,raw_knowledge_statement
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"However, RLHF is a complex and often unstable ..."
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,In this paper we introduce a new parameterizat...
1,Title/Abstract,No Subsection,\begin{abstract}\nWhile large-scale unsupervis...,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,"The resulting algorithm, which we call \textit..."
...,...,...,...,...,...,...
39,Discussion,No Subsection,"Learning from preferences is a powerful, scala...","\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...",Rather than coercing the preference learning p...
39,Discussion,No Subsection,"Learning from preferences is a powerful, scala...","\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","With virtually no tuning of hyperparameters, D..."
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...",Our initial results suggest that DPO policies ...
40,Discussion,No Subsection,\textbf{Limitations \& Future Work.} Our resul...,"\nLearning from preferences is a powerful, sca...","\nLearning from preferences is a powerful, sca...","Regarding evaluations, we find that the win ra..."


In [5]:
# Print all the NAs
na_rows = paper_df_validated[paper_df_validated['raw_knowledge_statement'].isna()]
print(f"Found {len(na_rows)} rows with NA knowledge statements:")
for idx, row in na_rows.iterrows():
    print(f"\nRow {idx}:")
    print(f"Paragraph: {row['paragraph']}")

Found 0 rows with NA knowledge statements:


### 2. Filter Bad Sentences

Not all extracted knowledge statements may be valid. We double check that the extracted statements meet our requirements for a proper probe.

##### 2.1 Filter Out Predominantly LaTeX Statements
This is to avoid figures, tables, and sentences that are dominated by LaTeX that prevents any suitable English target. LaTeX has various valid formats which can make evaluation tricky. There's also the chance that LaTeX introduces noise with regards to learnability. While OLMo has been trained on arxiv documents that include latex,it potentially may require further pre-training on mathematical notation and latex for the LLM to understand these statements.

In [6]:
# Filter out knowledge statements that are more than 50% LaTeX
def calculate_latex_percentage(text):
    """
    Calculate the percentage of LaTeX/mathematical content in a text string.
    
    Args:
        text (str): The text to analyze
        
    Returns:
        float: Percentage of text that is LaTeX/mathematical (0-100)
    """
    if pd.isna(text) or not text.strip():
        return 0.0
    
    import re
    
    total_chars = len(text)
    latex_chars = 0
    
    # Count LaTeX commands (backslash followed by letters)
    latex_commands = re.findall(r'\\[a-zA-Z]+', text)
    for cmd in latex_commands:
        latex_chars += len(cmd)
    
    # Remove LaTeX commands to avoid double counting
    text_without_commands = re.sub(r'\\[a-zA-Z]+', '', text)
    
    # Count non-alphabetic characters in the remaining text
    for char in text_without_commands:
        if not char.isalpha() and not char.isspace():
            latex_chars += 1
    
    # Calculate percentage
    latex_percentage = (latex_chars / total_chars) * 100 if total_chars > 0 else 0.0
    
    return latex_percentage

# Apply the filter
paper_df_validated['latex_percentage'] = paper_df_validated['raw_knowledge_statement'].apply(calculate_latex_percentage)

# Filter out statements with more than 50% LaTeX
latex_threshold = 70
paper_df_filtered = paper_df_validated[paper_df_validated['latex_percentage'] <= latex_threshold].copy()

# Report filtering results
total_before = len(paper_df_validated)
total_after = len(paper_df_filtered)
filtered_out = total_before - total_after

print(f"LaTeX filtering results:")
print(f"  Before filtering: {total_before} knowledge statements")
print(f"  After filtering: {total_after} knowledge statements")
print(f"  Filtered out: {filtered_out} statements with >{latex_threshold}% LaTeX content")

# Show all filtered statements
if filtered_out > 0:
    high_latex_statements = paper_df_validated[paper_df_validated['latex_percentage'] > latex_threshold]
    print(f"\nAll {filtered_out} filtered statements (high LaTeX content):")
    for idx, row in high_latex_statements.iterrows():
        print(f"  Row {idx}: LaTeX {row['latex_percentage']:.1f}%")
        print(f"    Statement: '{row['raw_knowledge_statement']}'")
        print()
        
paper_df_filtered.reset_index(drop=True, inplace=True)


LaTeX filtering results:
  Before filtering: 200 knowledge statements
  After filtering: 199 knowledge statements
  Filtered out: 1 statements with >70% LaTeX content

All 1 filtered statements (high LaTeX content):
  Row 26: LaTeX 70.4%
    Statement: 'With some algebra this leads to the optimization objective:
\begin{equation}\label{eq:AC}
    \max_{\pi_{\theta}}\mathbb{E}_{\pi_{\theta}(y\mid x)}\bigg[\underbrace{r_{\phi}(x, y) -\beta\log\sum_{y}\piref(y\mid x)\exp\left(\frac{1}{\beta}r_{\phi}(x, y)\right)}_{f(r_{\phi}, \piref, \beta)} - \underbrace{\beta\log\frac{\pi_{\theta}(y\mid x)}{\piref(y\mid x)}}_{\text{KL}}\bigg]
\end{equation}'



##### 2.2 Filter Out Sentences with References Not In Context

Some sentences contain references that are defined in a section of the paper that does not jointly appear during training since it's in a different section. Technically it may have appeared together if it's a small subsection since we join subsections that are small together, but they are at the very least far away. We filter out sentences that have such references.


In [7]:
import re
import pandas as pd

def find_undefined_references(sentence: str, subsection_text: str) -> list[str]:
    """
    Finds LaTeX references in a sentence that are not defined in the given subsection text.

    This function identifies all references formatted as \\ref{...} in the input sentence.
    It then checks for corresponding \\label{...} definitions within the subsection_text.
    
    Args:
        sentence (str): The sentence to check for references.
        subsection_text (str): The text of the subsection to check for labels.

    Returns:
        list[str]: A list of reference labels that are used in the sentence but not
                   defined in the subsection text. An empty list indicates all
                   references are defined locally.
    """
    # Find all references in the sentence, e.g., \ref{eq:RL} -> "eq:RL"
    references = re.findall(r'\\ref\{([^}]+)\}', sentence)
    if not references:
        return True

    # Find all defined labels in the subsection text, e.g., \label{eq:main_eq} -> "eq:main_eq"
    defined_labels = set(re.findall(r'\\label\{([^}]+)\}', subsection_text))

    # Identify references that are not defined within the subsection
    undefined_references = [ref for ref in references if ref not in defined_labels]

    if len(undefined_references) > 0:
        return False
    else:
        return True

keep = paper_df_filtered.apply(
    lambda row: find_undefined_references(row['raw_knowledge_statement'], row['subsection_text']),
    axis=1
)

# Report filtering results
total_before = len(paper_df_filtered)
dropped_statements = paper_df_filtered[~keep]
total_after = sum(keep)
filtered_out = total_before - total_after

print(f"Reference filtering results:")
print(f"  Before filtering: {total_before} knowledge statements")
print(f"  After filtering: {total_after} knowledge statements")
print(f"  Filtered out: {filtered_out} statements with undefined references")

# Show all dropped statements
if filtered_out > 0:
    print(f"\nAll {filtered_out} dropped statements (undefined references):")
    for idx, row in dropped_statements.iterrows():
        print(f"  Row {idx}: '{row['raw_knowledge_statement']}'")
        print()

paper_df_filtered = paper_df_filtered[keep].reset_index(drop=True)


Reference filtering results:
  Before filtering: 199 knowledge statements
  After filtering: 171 knowledge statements
  Filtered out: 28 statements with undefined references

All 28 dropped statements (undefined references):
  Row 65: '\textbf{Deriving the DPO objective.} We start with the same RL objective as prior work, Eq.~\ref{eq:RL}, under a general reward function $r$.'

  Row 66: 'Following prior work~\citep{peters2007reinforcement, peng2019advantage, korbak2022reinforcement, go2023aligning}, it is straightforward to show that the optimal solution to the KL-constrained reward maximization objective in Eq.~\ref{eq:RL} takes the form:
\begin{equation}\label{eq:op_policy}
    \pi_r(y\mid x) = \frac{1}{Z(x)}\piref(y\mid x)\exp\left(\frac{1}{\beta}r(x, y)\right),
\end{equation}%'

  Row 68: 'See Appendix \ref{app:derivation1} for a complete derivation.'

  Row 74: 'Substituting the reparameterization in Eq.~\ref{eq:main_eq} for $r^*(x,y)$ into the preference model Eq.~\ref{eq:bradley

##### 2.3 Filter Short Facts

In [8]:
# Filter out knowledge statements that are too short
min_length = 90
keep_length = paper_df_filtered['raw_knowledge_statement'].str.len() >= min_length

# Report filtering results
total_before = len(paper_df_filtered)
dropped_statements = paper_df_filtered[~keep_length]
total_after = sum(keep_length)
filtered_out = total_before - total_after

print(f"Length filtering results:")
print(f"  Before filtering: {total_before} knowledge statements")
print(f"  After filtering: {total_after} knowledge statements")
print(f"  Filtered out: {filtered_out} statements shorter than {min_length} characters")

# Show all dropped statements
if filtered_out > 0:
    print(f"\nAll {filtered_out} dropped statements (too short):")
    for idx, row in dropped_statements.iterrows():
        print(f"  Row {idx}: '{row['raw_knowledge_statement']}' (length: {len(row['raw_knowledge_statement'])})")

paper_df_filtered = paper_df_filtered[keep_length].reset_index(drop=True)


Length filtering results:
  Before filtering: 171 knowledge statements
  After filtering: 163 knowledge statements
  Filtered out: 8 statements shorter than 90 characters

All 8 dropped statements (too short):
  Row 56: 'In practice, the language model policy $\pi_\theta$ is also initialized to $\pisft$.' (length: 84)
  Row 84: 'We begin with by defining an equivalence relation between reward functions.' (length: 75)
  Row 106: 'Our experiments explore three different open-ended text generation tasks.' (length: 73)
  Row 134: 'This sweep includes 22 runs in total.' (length: 37)
  Row 137: 'This result is particularly notable for multiple reasons.' (length: 57)
  Row 144: 'DPO also achieves a higher maximum win rate compared to the best of $N$ baseline.' (length: 81)
  Row 147: 'Preferred-FT does not improve significantly over the SFT model.' (length: 63)
  Row 155: 'The results are presented in Table~\ref{tab:ood}.' (length: 49)


##### 2.4 Filter Unsuitable Probes

Sometimes the sentences that are extracted are trivial sentences that simply exist for transitonal purposes, or are rhetorical questions, etc. and essentially shouldn't have been tagged the first time. This was specified on the 1st extraction attempt, but we ensure that this is the case. We also check for extraction that failed and led to corrupted sentences i.e. cutoff in the middle. We specify the types of sentences that are *unsuitable* and make sure to filter out these sentences.

In [19]:
from tqdm import tqdm
from importlib import reload
reload(utils)
import json
# Evaluate knowledge statements for suitability
prompt = {}
prompt['system'] = """You are a meticulous, sharp, and detail-oriented evaluator.

# Instructions
You will be receiving clauses from an academic paper. Your task is to determine whether the clause is suitable for testing an LLM's factual recall. You will go about checking each of the following criteria for exclusion. If it satisfies any of these criteria, it is unsuitable. Note, one of the conditions ask if the sentence contains a a valid *target*. A target of a clause is a phrase, 1-2 words long, that is among the key, central information in that clause.

The clause is UNSUITABLE if it:
1. Mostly consists of mathematical expressions such that the only valid targets in the clause are mathematical expressions in LaTeX. 
 a. Plain english embedded within LaTeX commands, such as captions or italics, is SUITABLE and should NOT be considered unsuitable.
 b. If at least one valid English target can be found, it should NOT be considered unsuitable.
2. The clause only redirects to a section of the paper without adding any additional information (e.g., "This is discussed in Section 3.1"). If the clause also contains other information, it should NOT be considered unsuitable.
3. The clause is a rhetorical question.
4. The clause is not valid: it's unclear, cutoff in the middle, or shows displays of corruption.

# Output Format
Respond with JSON format with the following keys:
- "suitable": boolean (true/false)
- "unsuitable_condition": integer (1, 2, 3, or 4) or null if suitable
- "explanation": string"""

# Apply evaluation to the first knowledge statement only
print("Evaluating knowledge statement suitability...")

row = paper_df_filtered.iloc[93]
statement = row['raw_knowledge_statement']
user_prompt = f"Clause: {statement}"
print(user_prompt)
full_prompt = {
    'system': prompt['system'],
    'user': user_prompt
}

result = utils.query_llm(full_prompt, model='gpt-4.1', return_json=True, max_tokens=1000)
result = json.loads(result)
is_suitable = result['suitable']
unsuitable_condition = result.get('unsuitable_condition')
explanation = result.get('explanation')
print(f"First statement result:")
print(f"  Suitable: {is_suitable}")
print(f"  Unsuitable condition: {unsuitable_condition}")
print(f"  Explanation: {explanation}")


Evaluating knowledge statement suitability...
Clause: Since the added normalization term is only a function of the prefix $x$, $f(r; \piref, \beta)(x, y) $ is a reward function in the equivalence class of $r(x, y)$.
First statement result:
  Suitable: True
  Unsuitable condition: None
  Explanation: The clause contains valid English targets such as 'normalization term', 'prefix', 'reward function', and 'equivalence class'. While it includes some mathematical notation, it is not solely mathematical expressions, and the key information is conveyed in English. Therefore, it is suitable for testing factual recall.


In [12]:
from tqdm import tqdm
from importlib import reload
reload(utils)
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# Evaluate knowledge statements for suitability
prompt = {}
prompt['system'] = """You are a meticulous, sharp, and detail-oriented evaluator.

# Instructions
You will be receiving clauses from an academic paper. Your task is to determine whether the clause is suitable for testing an LLM's factual recall. You will go about checking each of the following criteria for exclusion. If it satisfies any of these criteria, it is unsuitable. Note, one of the conditions ask if the sentence contains a a valid *target*. A target of a clause is a phrase, 1-2 words long, that is among the key, central information in that clause.

The clause is UNSUITABLE if it:
1. Mostly consists of mathematical expressions such that the only valid targets in the clause are mathematical expressions in LaTeX. 
 a. Plain english embedded within LaTeX commands, such as captions or italics, is SUITABLE and should NOT be considered unsuitable.
 b. If at least one valid English target can be found, it should NOT be considered unsuitable.
2. The clause provides meta-commentary of the paper or points to a section of the paper without adding additional information. 
    a. For example, the clause can be reduced to "Our results raise several important questions for future work." or "In this section, we discuss our methodology." or "See Section 3.1." However, if the clause contains additional information than just this, it should NOT be considered unsuitable.
3. The clause is a rhetorical question.
4. The clause is a caption that describes what the table contains but doesn't actually discuss the contents i.e. doesn't actually add any information.
5. The clause is confusing: it's unclear, grammatically confusing, or cutoff in the middle.

# Output Format
Respond with JSON format with the following keys:
- "unsuitable_condition": integer (1, 2, 3, 4, or 5) or null if suitable
- "suitable": boolean (true/false)"""

def evaluate_statement(idx_row):
    idx, row = idx_row
    statement = row['raw_knowledge_statement']
    user_prompt = f"Context: {row['paragraph']}\n\nClause: {statement}"
    full_prompt = {
        'system': prompt['system'],
        'user': user_prompt
    }
    
    result = utils.query_llm(full_prompt, model='gpt-5-mini', return_json=True, max_tokens=5000)
    result = json.loads(result)
    is_suitable = result['suitable']
    unsuitable_condition = result.get('unsuitable_condition')
    
    return idx, is_suitable, unsuitable_condition

# Run evaluation 3 times and store separate results
print("Evaluating knowledge statement suitability (3 rounds)...")

# Print first statement
first_statement = paper_df_filtered.iloc[0]['raw_knowledge_statement']
print(f"Clause: {first_statement}")

# Initialize results for 3 rounds
suitability_results_1 = [None] * len(paper_df_filtered)
suitability_results_2 = [None] * len(paper_df_filtered)
suitability_results_3 = [None] * len(paper_df_filtered)
unsuitable_conditions_1 = [None] * len(paper_df_filtered)
unsuitable_conditions_2 = [None] * len(paper_df_filtered)
unsuitable_conditions_3 = [None] * len(paper_df_filtered)

for round_num in range(3):
    print(f"\nRound {round_num + 1}/3...")
    
    with ThreadPoolExecutor(max_workers=16) as executor:
        futures = {executor.submit(evaluate_statement, (idx, row)): idx 
                   for idx, row in paper_df_filtered.iterrows()}
        
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"Round {round_num + 1}"):
            idx, is_suitable, unsuitable_condition = future.result()
            original_idx = list(paper_df_filtered.index).index(idx)
            
            if round_num == 0:
                suitability_results_1[original_idx] = is_suitable
                unsuitable_conditions_1[original_idx] = unsuitable_condition
            elif round_num == 1:
                suitability_results_2[original_idx] = is_suitable
                unsuitable_conditions_2[original_idx] = unsuitable_condition
            else:
                suitability_results_3[original_idx] = is_suitable
                unsuitable_conditions_3[original_idx] = unsuitable_condition
        print(f"Round {round_num + 1} done...")

# Add all suitability columns
paper_df_filtered['is_suitable_1'] = suitability_results_1
paper_df_filtered['is_suitable_2'] = suitability_results_2
paper_df_filtered['is_suitable_3'] = suitability_results_3
paper_df_filtered['unsuitable_condition_1'] = unsuitable_conditions_1
paper_df_filtered['unsuitable_condition_2'] = unsuitable_conditions_2
paper_df_filtered['unsuitable_condition_3'] = unsuitable_conditions_3

# Count how many rounds marked each statement as unsuitable
paper_df_filtered['unsuitable_count'] = (~paper_df_filtered['is_suitable_1']).astype(int) + \
                                        (~paper_df_filtered['is_suitable_2']).astype(int) + \
                                        (~paper_df_filtered['is_suitable_3']).astype(int)

# Create final suitability column (unsuitable if at least 2 out of 3 rounds marked it as unsuitable)
paper_df_filtered['is_suitable'] = paper_df_filtered['unsuitable_count'] < 2

paper_df_suitable = paper_df_filtered[paper_df_filtered['is_suitable']].copy()

# Report filtering results
total_before = len(paper_df_filtered)
total_after = len(paper_df_suitable)
filtered_out = total_before - total_after

print(f"\nFinal suitability filtering results (at least 2 out of 3 rounds):")
print(f"  Before filtering: {total_before} knowledge statements")
print(f"  After filtering: {total_after} knowledge statements")
print(f"  Filtered out: {filtered_out} unsuitable statements")

Evaluating knowledge statement suitability (3 rounds)...
Clause: While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training.

Round 1/3...


Round 1: 100%|██████████| 163/163 [00:53<00:00,  3.03it/s]


Round 1 done...

Round 2/3...


Round 2: 100%|██████████| 163/163 [00:49<00:00,  3.29it/s]


Round 2 done...

Round 3/3...


Round 3: 100%|██████████| 163/163 [00:48<00:00,  3.33it/s]

Round 3 done...

Final suitability filtering results (at least 2 out of 3 rounds):
  Before filtering: 163 knowledge statements
  After filtering: 155 knowledge statements
  Filtered out: 8 unsuitable statements


In [13]:


# Show statements that got only one unsuitable
one_unsuitable = paper_df_filtered[paper_df_filtered['unsuitable_count'] == 1]
if len(one_unsuitable) > 0:
    print(f"\nStatements that got only one unsuitable (still kept):")
    for idx, row in one_unsuitable.iterrows():
        conditions = []
        if pd.notna(row['unsuitable_condition_1']) and not row['is_suitable_1']:
            conditions.append(f"Round 1: {row['unsuitable_condition_1']}")
        if pd.notna(row['unsuitable_condition_2']) and not row['is_suitable_2']:
            conditions.append(f"Round 2: {row['unsuitable_condition_2']}")
        if pd.notna(row['unsuitable_condition_3']) and not row['is_suitable_3']:
            conditions.append(f"Round 3: {row['unsuitable_condition_3']}")
        condition_str = ", ".join(conditions) if conditions else "None"
        print_wrapped(f"  Row {idx} ({condition_str}): '{row['raw_knowledge_statement']}'")


Statements that got only one unsuitable (still kept):
  Row 5 (Round 1: 2.0): 'Our experiments show that DPO can fine-tune LMs to align with human
preferences as well as or better than existing methods.'

  Row 11 (Round 2: 2.0): 'In other words, selecting the model's \emph{desired responses and
behavior} from its very wide \textit{knowledge and abilities} is crucial to building AI systems that
are safe, performant, and controllable \citep{ouyang2022training}.'

  Row 41 (Round 3: 2.0): 'We instead present a single stage policy learning approach that directly
optimizes a policy to satisfy preferences.'

  Row 71 (Round 2: 2.0): 'Now that we have the probability of human preference data in terms of the
optimal policy rather than the reward model, we can formulate a maximum likelihood objective for a
parametrized policy $\pi_\theta$.'

  Row 84 (Round 1: 2.0): 'It is easy to see that this is indeed an equivalence relation, which
partitions the set of reward functions into classes.'

  R

In [14]:
# Show all unsuitable statements with their conditions
if filtered_out > 0:
    unsuitable_statements = paper_df_filtered[~paper_df_filtered['is_suitable']]
    print(f"\nAll {filtered_out} unsuitable statements:")
    for idx, row in unsuitable_statements.iterrows():
        conditions = []
        if pd.notna(row['unsuitable_condition_1']):
            conditions.append(str(row['unsuitable_condition_1']))
        if pd.notna(row['unsuitable_condition_2']):
            conditions.append(str(row['unsuitable_condition_2']))
        if pd.notna(row['unsuitable_condition_3']):
            conditions.append(str(row['unsuitable_condition_3']))
        condition_str = ", ".join(conditions) if conditions else "None"
        print_wrapped(f"  Row {idx} (Conditions {condition_str}): '{row['raw_knowledge_statement']}'")


All 8 unsuitable statements:
  Row 19 (Conditions 2.0, 2.0, 2.0): 'In this paper, we show how to directly optimize a language
model to adhere to human preferences, without explicit reward modeling or reinforcement learning.'

  Row 25 (Conditions 2.0, 2.0): 'Our main contribution is Direct Preference Optimization (DPO), a
simple RL-free algorithm for training language models from preferences.'

  Row 34 (Conditions 2.0, 2.0, 2.0): 'Despite the appeal of using relative human preferences, fine-
tuning large language models with reinforcement learning remains a major practical challenge; this
work provides a theoretically-justified approach to optimizing relative preferences without RL.'

  Row 42 (Conditions 2.0, 2.0, 2.0): 'We review the RLHF pipeline in
\citeauthor{ziegler2020finetuning} (and later \citep{stiennon2022learning, bai2022training,
ouyang2022training}).'

  Row 82 (Conditions 2.0, 2.0, 2.0): 'In this section we will build the theory behind this
reparameterization, show tha

In [15]:
# Show all suitable statements
if total_after > 0:
    print(f"\nAll {total_after} suitable statements:")
    for idx, row in paper_df_suitable.iterrows():
        print_wrapped(f"  Row {idx}: '{row['raw_knowledge_statement']}'")


All 155 suitable statements:
  Row 0: 'While large-scale unsupervised language models (LMs) learn broad world knowledge and some
reasoning skills, achieving precise control of their behavior is difficult due to the completely
unsupervised nature of their training.'

  Row 1: 'Existing methods for gaining such steerability collect human labels of the relative
quality of model generations and fine-tune the unsupervised LM to align with these preferences,
often with reinforcement learning from human feedback (RLHF).'

  Row 2: 'However, RLHF is a complex and often unstable procedure, first fitting a reward model that
reflects the human preferences, and then fine-tuning the large unsupervised LM using reinforcement
learning to maximize this estimated reward without drifting too far from the original model.'

  Row 3: 'In this paper we introduce a new parameterization of the reward model in RLHF that enables
extraction of the corresponding optimal policy in closed form, allowing us to solv

##### 2.5 Extract Title

In [16]:
import re
paper_df_suitable.reset_index(drop=True, inplace=True)
paper_df_suitable['title'] = re.search(r'\\title{(.*?)}', paper).group(1) if re.search(r'\\title{(.*?)}', paper) else None

### 3. Extract Self-Contained, Atomic Probes
Given the original, source sentences from the paper, we break each sentence into the parts that presents a new fact.


In [17]:
def extract_context_and_sentence(row):
    # Extract context similar to the previous approach
    parts = row['section_text'].strip().split(row['raw_knowledge_statement'].strip())
    context_before = parts[0]
    
    if len(parts) > 1:
        remaining_text = parts[1]
        paragraph_end = remaining_text.find('\n\n')
        if paragraph_end != -1:
            rest_of_paragraph = remaining_text[:paragraph_end]
        else:
            rest_of_paragraph = remaining_text
        context = context_before + row['raw_knowledge_statement'].strip() + rest_of_paragraph
    else:
        context = context_before
    
    # Remove title line if present
    if context.startswith('\\title{'):
        lines = context.split('\n')
        # Find the end of the title (could span multiple lines)
        title_end = 0
        for i, line in enumerate(lines):
            if '}' in line:
                title_end = i + 1
                break
        context = '\n'.join(lines[title_end:]).strip()
    return context



In [57]:
import concurrent.futures

def extract_atomic_facts(first_row):
    """Extract atomic facts from a paragraph using LLM."""
    prompt = {}
    prompt['system'] = r"""You will be given two inputs, a section of an academic paper for context and a single sentence drawn from that section. Your task is to first extract facts from the sentence. For each fact, rewrite the fact into a self-contained, contextualized fact with an identifiable target at the end. Approach this task step-by-step as outlined below.

While you should use your expertise on this domain to handle and understand these texts, all information written into the facts *MUST* originate from the provided context or sentence. Do not add, infer, or correct information using your internal knowledge. Every detail should be traceable back to the source text. As you write and rewrite the facts, also make sure to accurately represent the knowledge in the original sentence without distortion. Strive to use phrasing as close as possible to the original text, but prioritize clarity and self-containment. Lastly, the facts should be written well and clearly so that they are easy to read.

### Step 1: Fact Extraction
Papers often interweave various pieces of knowledge together in a flow that begins from the very beginning until the very end. Eventhough each sentence is interwoven with other sentences, there is atomic knowledge that can be extracted. Identify and extract the atomic facts in the provided sentence. The extracted facts must not reference each other and be written separately.

One common error during this step is that the facts are not written independently of each other. For instance, Fact 1 may describe a procedure and Fact 2 may refer to it as "this procedure". If you see this, please go back and rewrite the facts so that they are written independently of each other.

### Step 2: Validating Targets
For each fact, take the last phrase in the sentence, and consider whether it is a valid target. We consider a target to be *valid* if 1) the target consists of 1-5 words 2) does not contain any special characters such as parenthesis or mathematical notation and 3) passes the "non-trivial" test. The target does not need to be the most core piece of information; it just needs to be "non-trivial".

This is the "non-trivial" test: if the fact was to be stripped of the target, it should be reasonable to infer the target based on the available context if one has acquired this knowledge. At the same time, it should be impossible to infer the target if one does not have this knowledge. For instance, in the sentence "Chain-of-thought reasoning can be elicited in large language models", the target is large language models. In this case, the target is valid because chain-of-thought reasoning is a property of large language models, making it non-trivial and inferrable from the available context but not completely trivial and obvious to a lay person who does not have this knowledge. 

If the last phrase in the fact is not a valid target, re-extract the fact as you did in the previous step such that a different phrase, that would be a valid target, becomes the target. If you can't think of any other way to extract the fact, discard the fact. Do not use a repetition strategy that repeats the target at the end after already having mentioned the target earlier in the fact 'i.e. ... {target}'. Discard the fact if you see this.

### Step 3: Contextualize Each Fact
Then, for each fact, add sufficient context so that the fact is precise. Specifically, use the *provided context* to supply whatever information is needed to make it clear what the fact is describing. For instance, "Do humans and GPT4 agree often with each other?" should be clarified into "In the paper '...', did humans and GPT4 often agree or disagree with each other during the evaluation of DPO?" if this notion was in the context of evaluating DPO in an academic paper.

Here are some additional guidelines:
- If there are equations, theorems, or defined variables being referenced, check the surrounding context and include the full definitions, equation, theorems, and givens that are being referenced. 
- There are often numerous experiments in a paper, and so supply enough experimental context so that the fact is clear about which experiment it is describing.
- If the answer is an acronym and the acronym appears frequently in the context, feel free to leave it as an acronym without defining it.
- The rewritten fact can be multiple sentences long.
- Be thorough in providing the necessary context.
- Please keep the facts separate and independent of each other.
 
To incorporate any mathematical definitions or theorems, use the following template (non-exhaustive):
- "$f$ is defined as... {question using $f$}"
- "Theorem 1 states that... {question using Theorem 1}"
- "Equation 1 is defined as... {question using Equation 1}"

### Final Instructions
Think carefully and critically through this task, following the step-by-step instructions outlined above, and provide your reasoning. Then, provide the final output, listing each fact and its corresponding target. I've provided some demonstrations below that can guide your reasoning and output format. 

### Demonstration 1
Context: "\\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}\n\\subsection{Can DPO scale to real preference datasets?}\nNext, we evaluate fine-tuning performance of DPO on summarization and single-turn dialogue. For summarization, automatic evaluation metrics such as ROUGE can be poorly correlated with human preferences~\citep{stiennon2022learning}, and prior work has found that fine-tuning LMs using PPO on human preferences to provide more effective summaries. We evaluate different methods by sampling completions on the test split of TL;DR summarization dataset, and computing the average win rate against reference completions in the test set."

Sentence: "We evaluate different methods by sampling completions on the test split of TL;DR summarization dataset, and computing the average win rate against reference completions in the test set."

Facts:
- Fact: "On summarization, the authors evaluate DPO’s fine-tuning performance against other methods by sampling completions on the test split of the dataset named TL;DR summarization", Target: "TL;DR summarization"
- Fact: "On summarization, the fine-tuning performance of DPO and other methods are evaluated by sampling completions on the test split of the TL;DR summarization dataset and computing the average win rate against the reference completions", Target: "reference completions"

### Demonstration 2
Context: "\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}\nWhile large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training. Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

Sentence: "Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

Facts:
- Fact: "To steer unsupervised language models, existing methods collect human labels", Target: "human labels"
- Fact: "Existing methods for steering unsupervised language models collect human labels of the relative quality of model generations", Target: "model generations"
- Fact: "Existing methods align unsupervised language models by fine-tuning on human preferences", Target: "preferences"
- Fact: "Existing methods for steering unsupervised language models via fine-tuning on human preferences often use reinforcement learning from human feedback", Target: "human feedback"
"""
    # Split by raw knowledge statement to get everything before it
    context = extract_context_and_sentence(first_row)
    # print("Context: ", context)
    # print("--------------------------------")
    prompt['user'] = f"""### Context\n\\title{{{first_row['title']}}}\n\n\section{{{first_row['section'].strip()}}}\n{context}\n\n### Sentence\n{first_row['raw_knowledge_statement'].strip()}"""
    #print(prompt['user'])
    if len(first_row['raw_knowledge_statement'].strip()) < 50:
        return None, None, None
    output1 = utils.query_llm(prompt, model='gpt-5-mini')
    # output2 = utils.query_llm(prompt, model='gpt-5-mini')
    # output3 = utils.query_llm(prompt, model='gpt-5-mini')
    return output1 #, output2, output3
# # Process all rows

# with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
#     raw_extracted_facts = list(tqdm(executor.map(extract_atomic_facts, [paper_df_suitable.iloc[i] for i in range(len(paper_df_suitable))]), total=len(paper_df_suitable)))

# # Separate the three outputs into different columns
# output1_list = [result[0] if result is not None else None for result in raw_extracted_facts]
# output2_list = [result[1] if result is not None else None for result in raw_extracted_facts]
# output3_list = [result[2] if result is not None else None for result in raw_extracted_facts]

# # Store in dataframe
# paper_df_suitable['raw_extracted_facts_1'] = output1_list
# paper_df_suitable['raw_extracted_facts_2'] = output2_list
# paper_df_suitable['raw_extracted_facts_3'] = output3_list

# print(f"Processed {len(raw_extracted_facts)} rows")
# print(f"Sample result 1: {raw_extracted_facts[0][0] if raw_extracted_facts[0] is not None else None}")
# print(f"Sample result 2: {raw_extracted_facts[0][1] if raw_extracted_facts[0] is not None else None}")
# print(f"Sample result 3: {raw_extracted_facts[0][2] if raw_extracted_facts[0] is not None else None}")

# Process just one row for testing
# 
import random

# Process six random rows in parallel
random_indices = [1, 5, 25, 64, 83, 86]
random_rows = [paper_df_suitable.iloc[i] for i in random_indices]

with concurrent.futures.ThreadPoolExecutor(max_workers=6) as executor:
    results = list(executor.map(extract_atomic_facts, random_rows))

print(f"Processed 4 random rows")
for i, (idx, row, result) in enumerate(zip(random_indices, random_rows, results)):
    print(f"\n--- Row {i+1} (index {idx}) ---")
    print(f"Title: {row['title']}")
    print("Context: ", extract_context_and_sentence(row))
    print(f"Sentence: {row['raw_knowledge_statement']}")
    print(f"Result: {result}")

# with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
#     raw_extracted_facts = list(tqdm(executor.map(extract_atomic_facts, [paper_df_suitable.iloc[i] for i in range(len(paper_df_suitable))]), total=len(paper_df_suitable)))

# # Store in dataframe
# paper_df_suitable['raw_extracted_facts'] = raw_extracted_facts

# print(f"Processed {len(raw_extracted_facts)} rows")
# print(f"Sample result: {raw_extracted_facts[0] if raw_extracted_facts[0] is not None else None}")


# ### Example 2
# Context: "The general DPO pipeline is as follows: 1) Sample completions $y_1, y_2 \sim \piref(\cdot \mid x)$ for every prompt $x$, label with human preferences to construct the offline dataset of preferences $\mathcal{D} = \{x^{(i)}, y_w^{(i)}, y_l)^{(i)}\}_{i=1}^N$ and 2) optimize the language model $\pi_\theta$ to minimize $\mathcal{L}_\text{DPO}$ for the given $\piref$ and $\mathcal{D}$ and desired $\beta$. The base reference policy can be referred to as $\piref$, and the initial SFT model as $\pisft$. In practice, one would like to reuse preference datasets publicly available, rather than generating samples and gathering human preferences. Since the preference datasets are sampled using $\pisft$, we initialize $\piref = \pisft$ whenever available. However, when $\pisft$ is not available, we initialize $\piref$ by maximizing likelihood of preferred completions ${(x, y_w)}$, that is, ${\piref = \argmax_{\pi}\mathbb{E}_{x, y_w \sim \mathcal{D}}\left[\log \pi(y_w \mid x)\right]}$. This procedure helps mitigate the distribution shift between the true reference distribution which is unavailable, and $\piref$ used by DPO."

# Sentence: "Since the preference datasets are sampled using $\pisft$, we initialize $\piref = \pisft$ whenever available."

# Facts:
# - Fact: "In the DPO pipeline, the base reference policy $\piref$ is initialized to the available SFT model", Target: "SFT model

# ### Example 1
# Context: "\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}\nWhile large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training. Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

# Sentence: "Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

# Facts:
# - Fact: "In the paper 'Direct Preference Optimization: Your Language Model is Secretly a Reward Model', the authors remark that to steer unsupervised language models, existing methods collect human labels", Target: "human labels"
# - Fact: "In the paper 'Direct Preference Optimization: Your Language Model is Secretly a Reward Model', the authors remark that existing methods for steering unsupervised language models collect human labels of the relative quality of model generations", Target: "model generations"
# - Fact: "In the paper 'Direct Preference Optimization: Your Language Model is Secretly a Reward Model', the authors remark that existing methods align unsupervised language models by fine-tuning on human preferences", Target: "preferences"
# - Fact: "In the paper 'Direct Preference Optimization: Your Language Model is Secretly a Reward Model', the authors remark that existing methods for steering unsupervised language models via fine-tuning on human preferences often use reinforcement learning from human feedback", Target: "human feedback"


# ### Example 3
# Context: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training."

# Sentence: "While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training." 

# Targets:
# - "world knowledge"
# - "reasoning skills"
# - "precise control"
# - "difficult
# - "behavior"
# - "unsupervised training"

# Rewritten Facts:
# - "world knowledge": "Large-scale unsupervised LMs learn broad world knowledge" 
# - "reasoning skills": "Large-scale unsupervised LMs learn reasoning skills" 
# - "precise control": "Due to the completely unsupervised nature of the training of large-scale unsupervised language models, their behavior is difficult to precisely control"
# - "difficult": "Due to the completely unsupervised nature of the training of large-scale unsupervised language models, achieving precise control of their behavior is difficult"
# - "behavior": "Due to the completely unsupervised nature of the training of large-scale unsupervised language models, it is difficult to achieve precise control of their behavior"
# - "unsupervised training": "Achieving precise control of the behavior of unsupervised language models is difficult due to their unsupervised training"

### Step 4: Filter
# Is the fact clear? If it's not actually obvious from the sentence, we shouldn't have extracted the target. If there are many targets, consider if some of the targets are insignificant. Considering these aspects, filter out some targets and facts.

Processed 4 random rows

--- Row 1 (index 1) ---
Title: Direct Preference Optimization: Your Language Model is Secretly a Reward Model
Context:  \begin{abstract}
While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training.
Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF).
However, RLHF is a complex and often unstable procedure, first fitting a reward model that reflects the human preferences, and then fine-tuning the large unsupervised LM using reinforcement learning to maximize this estimated reward without drifting too far from the original model.
In this paper we introduce a new parameterization of the reward model in RLHF that enab

You will be given two inputs, a section of an academic paper for context and a single sentence drawn from that section. Your task is to first extract questions from the sentence with a clear 1-3 word answer. Then turn each question into a self-contained question that's interpretable on its own. Lastly, for each question, rewrite the question into a sentence of cloze form where the answer is exactly at the end. Approach this task step-by-step as outlined below.

### Overarching Instructions
At every step, upheld these core principles:
- Source Grounding: While you should use your expertise on this domain to handle and understand these texts, all information written into the questions and answers *MUST* originate from the provided context or sentence. Do not add, infer, or correct information using your internal knowledge. Every detail should be traceable back to the source text. 
- Preserve Meaning: As you write and rewrite the questions, accurately represent the knowledge in the original sentence without distortion. Strive to use phrasing as close as possible to the original text, but prioritize clarity and self-containment.
- Writing Quality: The questions should be written well and clearly so that they are easy to read.

### Step 1: Question Extraction
Sentences in academic papers often interweave various pieces of knowledge together. Identify and extract the main, atomic facts in the provided sentence; a fact should be what's merely stated by the sentence or a direct implication of the sentence. Write questions for these facts that underly the sentence. 
- Each question should have a a clear, single answer and NOT multiple valid answers. 
- The answer to the question must be 1-3 words and NOT involve any special characters or mathematical notation.
- The extracted questions must NOT reference each other; they should be self-contained, written independently of each other

### Step 2: Contextualize Each Question
Then, for each question, add enough context so that the rewritten question is precise, self-contained, and interpretable on its own. The rewritten question doesn't need to be a single sentence; if it's better to break it up, write sentences to set the context, and then a sentence to state the question. In either case, use the *provided context* to supply whatever information is needed to make the question independent and standalone. For instance, "Do humans and GPT4 agree often with each other?" should be clarified into "In the paper '...', did humans and GPT4 often agree or disagree with each other during the evaluation of DPO?" if this notion was in the context of evaluating DPO in an academic paper. Be thorough in your examination to provide the necessary context.

Look out for these cases in particular:
- *Math Contextualization:* If there are equations, theorems, or definde variables e.g. $f$ that are being referenced, check the surrounding context and include the full definitions, equation, theorems, and givens that are being referenced to fully contextualize the math. Citing the reference is not enough, but the original definition, equation, theorems, and givens should be included into the question.
- *Experimental Contextualization:* There are often numerous experiments in a paper, and so supply enough experimental context so that the question is precise and interpretable on its own. 
- *Acronyms*: If the answer is an acronym and the acronym appears frequently in the context, feel free to leave it as an acronym without defining it.

For every question, start with this template for contextualizing:
- "In the paper '{title}', ..."
- "According to the paper '{title}',..."

This is just a starting point and a non-exhaustive template, and you should use your own judgement to supply enough context so that the question is clear and interpretable on its own.

### Step 3: Turn each Question into Cloze Form
At the end, turn the question and answer into a cloze form sentence in which the answer is placed at the very end of the sentence.

### Final Instructions
Think carefully and critically through this task, following the step-by-step instructions outlined above, and provide your reasoning. Then, provide the final output, listing each question and its corresponding answer. I've provided some demonstrations below that can guide your reasoning and output format. 

### Demonstration 1
Context: "\\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}\n\\subsection{Can DPO scale to real preference datasets?}\nNext, we evaluate fine-tuning performance of DPO on summarization and single-turn dialogue. For summarization, automatic evaluation metrics such as ROUGE can be poorly correlated with human preferences~\citep{stiennon2022learning}, and prior work has found that fine-tuning LMs using PPO on human preferences to provide more effective summaries. We evaluate different methods by sampling completions on the test split of TL;DR summarization dataset, and computing the average win rate against reference completions in the test set."

Sentence: "We evaluate different methods by sampling completions on the test split of TL;DR summarization dataset, and computing the average win rate against reference completions in the test set."

Cloze Sentences:
- "In the paper 'Direct Preference Optimization: Your Language Model is Secretly a Reward Model', the authors evaluate DPO’s fine-tuning performance against other methods on summarization by sampling completions on the test split of the dataset named TL;DR summarization", Answer: "TL;DR summarization"
- "In the paper 'Direct Preference Optimization: Your Language Model is Secretly a Reward Model', the fine-tuning performance of DPO and other methods on summarization are evaluated by sampling completions on the test split of the TL;DR summarization dataset and computing the average win rate against the reference completions", Answer: "reference completions"

### Demonstration 2
Context: "\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}\nWhile large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training. Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

Sentence: "Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

Cloze Sentences:
- "According to the paper 'Direct Preference Optimization: Your Language Model is Secretly a Reward Model', to steer unsupervised language models, existing methods collect human labels", Answer: "human labels"
- "According to the paper 'Direct Preference Optimization: Your Language Model is Secretly a Reward Model', existing methods for steering unsupervised language models collect human labels of the relative quality of model generations", Answer: "model generations"
- "According to the paper 'Direct Preference Optimization: Your Language Model is Secretly a Reward Model', existing methods align unsupervised language models by fine-tuning on human preferences", Answer: "preferences"
- "According to the paper 'Direct Preference Optimization: Your Language Model is Secretly a Reward Model', existing methods for steering unsupervised language models via fine-tuning on human preferences often use reinforcement learning from human feedback", Answer: "human feedback"
"""

2.  Clarify pronouns. Check the sentence for any terms that are being referred to by a pronoun and is undefined otherwise. Search the surrounding context and include the context of the term that is being referenced or being referred to e.g. "This equation". 
3.  Clarify Context-Dependent Terms. Named entities e.g. theorems, equations, proper nouns do not need to be clarified. But, if there are unnamed or context-specific terms (e.g. $f$, the model, the loss), please clarify the full context of the term. For instance, "the gradient" in a sentence might be referring to the general idea of a gradient or the gradient of some specific object that was mentioned several sentences ago. 

In [ ]:

# To incorporate any mathematical definitions or theorems, use the following template (non-exhaustive):
# - "$f$ is defined as... {question using $f$}"
# - "Theorem 1 states that... {question using Theorem 1}"
# - "Equation 1 is defined as... {question using Equation 1}"
import concurrent.futures
reload(utils)
def extract_atomic_facts(first_row):
    """Extract atomic facts from a paragraph using LLM."""
    prompt = {}
    prompt['system'] = r"""You will be given two inputs, a section of an academic paper for context and a single sentence drawn from that section. Your task is to first extract questions from the sentence with a clear 1-5 word answer. Then turn each question into a self-contained question that's interpretable on its own. Approach this task step-by-step as outlined below.

While you should use your expertise on this domain to handle and understand these texts, all information written into the questions and answers *MUST* originate from the provided context or sentence. Do not add, infer, or correct information using your internal knowledge. Every detail should be traceable back to the source text. As you write and rewrite the questions, also make sure to accurately represent the knowledge in the original sentence without distortion. Strive to use phrasing as close as possible to the original text, but prioritize clarity and self-containment. Lastly, the questions should be written well and clearly so that they are easy to read.

### Instructions
Papers often interweave various pieces of knowledge together in a flow that begins from the very beginning until the very end. While each sentence is interwoven with other sentences, there is atomic knowledge that can be extracted from a particular sentence. Write questions that tests for the knowledge presented by the sentence, following these criteria:
- The answer to the question must be a coherent phrase, 1-3 words long, and should be as compact as possible (e.g., prefer "reward function" over "normalizes the reward function").
- The answer must *NOT* involve any *special characters* or *mathematical notation*. Again, any phrase that contains math should not be used as the answer.
- The question should have a a clear, single answer and *NOT* multiple valid answers. 
- Each question should be written separately and independently of the other questions, so don't reference other questions in the same question.

### Demonstration 1
Context: "\\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}\n\\subsection{Can DPO scale to real preference datasets?}\nNext, we evaluate fine-tuning performance of DPO on summarization and single-turn dialogue. For summarization, automatic evaluation metrics such as ROUGE can be poorly correlated with human preferences~\citep{stiennon2022learning}, and prior work has found that fine-tuning LMs using PPO on human preferences to provide more effective summaries. We evaluate different methods by sampling completions on the test split of TL;DR summarization dataset, and computing the average win rate against reference completions in the test set."

Sentence: "We evaluate different methods by sampling completions on the test split of TL;DR summarization dataset, and computing the average win rate against reference completions in the test set."

Questions:
- "The authors evaluate DPO’s fine-tuning performance against other methods on summarization by sampling completions on the test split of what dataset?", Answer: "TL;DR summarization"
- "The fine-tuning performance of DPO and other methods on summarization are evaluated by sampling completions on the test split of the TL;DR summarization dataset and computing the average win rate against what?", Answer: "the reference completions"

### Demonstration 2
Context: "\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}\nWhile large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training. Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

Sentence: "Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

Questions:
- "What do existing methods collect to steer unsupervised language models, ?", Answer: "human labels"
- "Existing methods for steering unsupervised language models collect human labels of the quality of what?", Answer: "relative quality of model generations"
- "Existing methods align unsupervised language models by fine-tuning on what?", Answer: "human preferences"
- "Existing methods for steering unsupervised language models via fine-tuning on human preferences often use what?", Answer: "RLHF"
"""
    contextualize_prompt = {}
    contextualize_prompt['system'] = r"""You will be given two inputs, a section of an academic paper for context, a single sentence drawn from that section, and a list of questions extracted from the sentence as well as their corresponding answers. Your task is to then turn each question into a self-contained, precise question. Approach this task step-by-step as outlined below.

While you should use your expertise on this domain to handle and understand these texts, all information written into the questions and answers *MUST* originate from the provided context or sentence. Do not add, infer, or correct information using your internal knowledge. Every detail should be traceable back to the source text. As you write and rewrite the questions, also make sure to accurately represent the knowledge in the original sentence without distortion. Strive to use phrasing as close as possible to the original text, but prioritize clarity and self-containment. Lastly, the questions should be written well and clearly so that they are easy to read.
    
### Instructions
The overall goal of this task is to make the questions clear by incorporating the relevant context. This ensures the question is unambiguous and that it directly tests the knowledge expressed in the source sentence.

For each question:
1.  Rewrite the question so that it starts with one of the following templates. 
    - "In the paper '{title}', ..."
    - "According to the paper '{title}',..."
    - "In the paper '{title}', the authors remark that..."
    - "In the paper '{title}', the authors state that..."
    - "According to the paper '{title}', prior work has..."
    - "In the theoretical analysis of the paper "{title}"..."
    - "In the paper '{title}', the results suggest that..."
    This is a non-exhaustive list of templates, and you should use your own judgement to choose the most appropriate template or modify the template to fit the sentence.
2.  Add sufficient context. Specifically, use the *provided context* to supply whatever information is needed to make the question self-contained and unambiguous. For instance, "Do humans and GPT4 agree often with each other?" should be clarified into "In the paper '...', did humans and GPT4 often agree or disagree with each other during the evaluation of DPO?" if this notion was in the context of evaluating DPO in an academic paper. The goal is to ensure someone reading just the question would understand exactly what is being asked without needing additional context.
3.  Clarify pronouns and referential terms. Check the sentence for pronouns (it, this, that, these, those) or demonstrative phrases (this equation, that method, these results) that refer to entities not explicitly defined within the sentence itself. Search the surrounding context to identify what these terms reference, then incorporate that clarifying information into the question to make it self-contained.
4.  Clarify Context-Dependent Terms. Named entities (e.g., theorems, equations, proper nouns) do not need clarification. However, if there are unnamed or context-specific terms (e.g., $f$, "the model", "the loss"), clarify their full context. For instance, "the gradient" might refer to the general concept of a gradient or to the gradient of a specific function mentioned earlier in the context.
5.  Disambiguate experiments. There are often numerous experiments in a paper, and so supply enough experimental context so that the question is about which experiment the question is asking about. 
6.  Handle acronyms. If the answer is an acronym and the acronym appears frequently in the context, feel free to leave it as an acronym without defining it.
7.  Do not leak the answer. Please make sure that *the answer is not revealed* in the question. The answer should never appear in the question.
8.  Keep the questions separate.
9.  Do not change the answer. Minor grammatical adjustments to the answer are allowed only if necessary to fit the restructured question (e.g., adjusting verb tense, dropping pronouns like "it").
10. Refine Question. The rewritten question can be broken up into multiple sentences if the question becomes verbose. Make sure the question is written clearly and grammatically correct. Do not put any of the context in parenthesis or explained by "i.e.".

Think carefully and critically through this task, following the step-by-step instructions outlined above. Then, provide the final output, listing each question and its corresponding answer."""
    cloze_prompt = {}
    cloze_prompt['system'] = r"""### Instructions
You will be given a list of pairs of questions and answers. Turn each question and answer into a statement in which the answer is placed at the very end. 
- Incorporate the answer naturally and adjust the answer if necessary.
- The answer must be at the *very end* of the the last sentence.
- The paraphrasing should be natural and flow well, with the answer naturally appearing at the very end. Avoid awkward repetition or forced placement of the answer. If restructuring the question to end with the answer feels unnatural or forced, discard that question entirely.
- Ensure the statement is grammatically correct and well-written. Adjust the answer only if necessary to maintain grammatical correctness.
- Preserve all information and context from the original question when converting to cloze form; the statement form should very similarly resemble the original question.

### Demonstration

Question: Within the DPO theoretical section, the projection operator f normalizes the reward function by subtracting a term involving the policy’s partition function. Which mathematical function of the partition function is used for this normalization?  
Answer: logarithm

Statement: Within the DPO theoretical section, the projection operator f normalizes the reward function by subtracting a term involving the policy’s partition function. The mathematical function of the partition function used for this normalization is the logarithm.

### Output Format 
List of (answer, statement)."""
    quality_control_prompt = {}
    quality_control_prompt['system'] = r"""You are a meticulous Quality Control Assistant. Your task is to review and refine statements that have been extracted from an academic paper. You will be given a list of '(answer, statement)' pairs and the original context from which they were derived. Your task is to apply a rigorous checklist to each pair, refining it based on a provided quality control checklist.

### Quality Control Checklist

For each '(answer, statement)' pair, verify the following:

1. Answer Leakage: 
- The statement may contain the answer in the question before it appears in the end. 
- Action: Adjust the statement as minimally as necessary such that the answer is not leaked in the middle of the statement and it only appears at the very end of the statement.

2. LaTeX Formatting
- All mathematical expressions and notations **MUST** be written in LaTeX, enclosed in '$' or '$$' delimiters.
- Do *NOT* use unicode mathematical characters (e.g., use '\\pi', not 'π').
- Do *NOT* use unnecessary styling commands like '\\displaystyle'.
- Ensure LaTeX syntax matches the style of the original context (e.g., '( ... )' or '$ ... $').
- Action: Rewrite the math expressions and statements so they can be written in LaTeX, keeping the rest of the statement the same, correcting any and all formatting errors related to mathematical notation.

3. Answer Adjustment:
- Sometimes the answer starts with "the". In this case, leave the statement as is but strip "the" from the beginning of the answer.

In all your adjustments, do not change the structure, content, or shape of the statement. All of these adjustments should be word-level or character-level adjustments.

### Output Format
After your review, provide a list of the refined '(answer, statement)' pairs that have passed all checks in JSON format.
- "pairs": list of dicts i.e ("target": target, "statement": statement)"""

    # Split by raw knowledge statement to get everything before it
    context = extract_context_and_sentence(first_row)
    print("Context: ", context)
    print("--------------------------------")
    prompt['user'] = f"""### Title\n{first_row['title']}\n### Context\n{first_row['subsection_text']}\n\n### Sentence\n{first_row['raw_knowledge_statement'].strip()}"""
    output1 = utils.query_llm(prompt, model='gpt-5-mini')   
    contextualize_prompt['user'] = f"""### Title\n{first_row['title']}\n### Context{first_row['subsection']}\n{context}\n\n### Sentence\n{first_row['raw_knowledge_statement'].strip()}\n\n### Questions\n{output1}"""
    output2 = utils.query_llm(contextualize_prompt, model='gpt-5-mini')
    cloze_prompt['user'] = f"""### Questions\n{output2}"""
    output3 = utils.query_gpt(cloze_prompt, system_prompt_included=True, model='gpt-5-mini', reasoning_effort='medium')
    quality_control_prompt['user'] = f"""### Context\n{first_row['subsection_text']}\n\n### Sentence\n{first_row['raw_knowledge_statement'].strip()}### Pairs: {output3}"""
    output4 = utils.query_gpt(quality_control_prompt, system_prompt_included=True, model='gpt-5-mini', reasoning_effort='low', return_json=True)
    return output1 , output2, output3, output4
# # Process all rows

# with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
#     raw_extracted_facts = list(tqdm(executor.map(extract_atomic_facts, [paper_df_suitable.iloc[i] for i in range(len(paper_df_suitable))]), total=len(paper_df_suitable)))

# # Separate the three outputs into different columns
# output1_list = [result[0] if result is not None else None for result in raw_extracted_facts]
# output2_list = [result[1] if result is not None else None for result in raw_extracted_facts]
# output3_list = [result[2] if result is not None else None for result in raw_extracted_facts]

# # Store in dataframe
# paper_df_suitable['raw_extracted_facts_1'] = output1_list
# paper_df_suitable['raw_extracted_facts_2'] = output2_list
# paper_df_suitable['raw_extracted_facts_3'] = output3_list

# print(f"Processed {len(raw_extracted_facts)} rows")
# print(f"Sample result 1: {raw_extracted_facts[0][0] if raw_extracted_facts[0] is not None else None}")
# print(f"Sample result 2: {raw_extracted_facts[0][1] if raw_extracted_facts[0] is not None else None}")
# print(f"Sample result 3: {raw_extracted_facts[0][2] if raw_extracted_facts[0] is not None else None}")

# Process just one row for testing
# 
# import random

# # Process six random rows in parallel
# random_indices = [1, 5, 25, 64, 83, 86]
# random_rows = [paper_df_suitable.iloc[i] for i in random_indices]

# with concurrent.futures.ThreadPoolExecutor(max_workers=6) as executor:
#     results = list(executor.map(extract_atomic_facts, random_rows))

# print(f"Processed 4 random rows")
# for i, (idx, row, result) in enumerate(zip(random_indices, random_rows, results)):
#     print(f"\n--- Row {i+1} (index {idx}) ---")
#     print(f"Title: {row['title']}")
#     print("Context: ", extract_context_and_sentence(row))
#     print(f"Sentence: {row['raw_knowledge_statement']}")
#     print("--------------------------------")
#     print(f"## Questions\n{result[0]}")
#     print("--------------------------------")
#     print(f"## Contextualized Questions\n{result[1]}")
#     print("--------------------------------")
#     print(f"## Final Cloze Sentences\n{result[2]}")
#     print("--------------------------------")
#     print(f"## Cleaned Statements\n{result[3]}")
#     print("--------------------------------")

with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
    raw_extracted_facts = list(tqdm(executor.map(extract_atomic_facts, [paper_df_suitable.iloc[i] for i in range(len(paper_df_suitable))]), total=len(paper_df_suitable)))

# Store in dataframe
paper_df_suitable['raw_extracted_facts'] = raw_extracted_facts

print(f"Processed {len(raw_extracted_facts)} rows")
print(f"Sample result: {raw_extracted_facts[0] if raw_extracted_facts[0] is not None else None}")


In [83]:
paper_df_suitable['validated_atomic_pairs'] = [json.loads(raw_extracted_facts[3])['pairs'] for raw_extracted_facts in raw_extracted_facts]

In [84]:
paper_df_suitable['validated_atomic_pairs']

0      [{'target': 'broad world knowledge', 'statemen...
1      [{'target': 'human labels', 'statement': 'In t...
2      [{'target': 'complex and unstable', 'statement...
3      [{'target': 'new parameterization', 'statement...
4      [{'target': 'Direct Preference Optimization', ...
                             ...                        
150    [{'target': 'language model policies', 'statem...
151    [{'target': 'virtually no tuning', 'statement'...
152    [{'target': 'PPO models', 'statement': 'Accord...
153    [{'target': 'win rates', 'statement': 'Accordi...
154    [{'target': 'human preferences', 'statement': ...
Name: validated_atomic_pairs, Length: 155, dtype: object

In [78]:
# Print 25 random outputs from the extracted facts
import random

# Get valid indices (where raw_extracted_facts is not None)
valid_indices = [i for i, facts in enumerate(raw_extracted_facts) if facts is not None]

# Sample 25 random indices
sample_indices = sorted(random.sample(valid_indices, min(20, len(valid_indices))))

print(f"Showing {len(sample_indices)} random outputs from extracted facts:\n")
print("=" * 80)

for i, idx in enumerate(sample_indices, 1):
    row = paper_df_suitable.iloc[idx]
    facts = raw_extracted_facts[idx]
    
    print(f"\n{i}. Row {idx}")
    print(f"Title: {row['title']}")
    print(f"Subsection: {row['subsection']}")
    print(f"Original sentence: {row['raw_knowledge_statement']}")
    print(f"Extracted facts: {facts}")
    print("-" * 50)


Showing 20 random outputs from extracted facts:


1. Row 8
Title: Direct Preference Optimization: Your Language Model is Secretly a Reward Model
Subsection: No Subsection
Original sentence: However, these models are trained on data generated by humans with a wide variety of goals, priorities, and skillsets.
Extracted facts: ('Step 1 — Extracted questions (short form) and answers:\n- "Who generated the training data?", Answer: humans\n- "What are the models trained on?", Answer: human generated data\n- "The human-generated data has a wide variety of what?", Answer: goals, priorities, skillsets\n\nStep 2 — Self-contained questions and answers:\n- "The language models are trained on data generated by whom?", Answer: humans\n- "What kind of data are the language models trained on?", Answer: human generated data\n- "The humans who generated the training data have a wide variety of what?", Answer: goals, priorities, skillsets', "According to the paper 'Direct Preference Optimization: Your La

### [OLD] 4. Filter and Refine Probes

In [ ]:
import concurrent.futures
 
def validate_atomic_facts(previous_output):
    """Extract atomic facts from a paragraph using LLM."""
    prompt = {}
    prompt['system'] = """You are a quality control assistant. Your task is to review and refine the output of a previous process. The original task was to deconstruct a sentence from a paper into self-contained, atomic facts that ends with their corresponding targets.

## Instructions
Approach this task carefully, following the step-by-step guidelines provided below and provide your reasoning before the final output.

First, extract the finalized list of `(target, rewritten_sentence)` pairs from the output of the previous process.

Then, for each pair of target and fact, carefully perform the following checks.
- *Target Validity:*  Does the fact properly end with its corresponding target? Does the target only appear once in the fact?
- *Fact Quality:* Is the writing of the fact natural? Is it fully understandable on its own?

If any pairs fail the checks, discard them.

Lastly, output the filtered list of `(target, rewritten_sentence)` pairs. Copy the facts exactly as they are, including any LaTeX formatting. Make sure that any mathematical expressions or notations are written in the same LaTeX format as provided."""
    prompt['user'] = f"""# Output of Previous Process\n\n{previous_output}"""
    
    response = utils.query_llm(prompt, model='o4-mini', system_prompt_included=True)
    return response
    
def format_atomic_facts(previous_output):
    prompt = {}
    prompt['system'] = """Format the final list of targets and their corresponding facts into JSON with the following key. Copy the facts exactly as they are, including any LaTeX formatting.
- "pairs": list of dicts i.e ("target": target, "rewritten_fact": rewritten fact)"""
    prompt['user'] = f"""### List of Facts\n\n{previous_output}"""
    
    response = utils.query_gpt(prompt, model='o4-mini', reasoning_effort='low', return_json=True, system_prompt_included=True)
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        print(f"Failed to parse JSON: {response}")
        return None
    
print(paper_df_suitable.iloc[80]['raw_extracted_facts'])
filtering = validate_atomic_facts(paper_df_suitable.iloc[80]['raw_extracted_facts'])
print(filtering)
formatted_facts = format_atomic_facts(filtering)
print(formatted_facts)
    

Let's proceed step by step as instructed.

---

## Step 1: Target Extraction

Sentence:  
"We can alternatively view Theorem~\ref{thm:main} as specifying exactly which reward function within each equivalence class the DPO reparameterization selects, that is, the reward function satisfying:
\begin{equation}\label{eq:lag_p}
     \sum_{y}\underbrace{\piref(y\mid x)\exp\left(\frac{1}{\beta}r(x, y)\right)}_{=\pi(y\mid x)\text{, using Thm.~\ref{thm:main} reparam.}} = 1,
\end{equation}
i.e., $\pi(y\mid x)$ is a valid distribution (probabilities are positive and sum to 1)."

Key, central information:
- "reward function"
- "equivalence class"
- "DPO reparameterization"
- "valid distribution"

Let's check which of these are central. The sentence is about Theorem~\ref{thm:main} and how it specifies which reward function (within an equivalence class) is selected by the DPO reparameterization, namely the one that makes $\pi(y|x)$ a valid distribution. The main predicates are "specifying", "selects"

### 4. [NEW] Refine

In [129]:
paper_df_suitable.iloc[86]['raw_knowledge_statement']

'We can alternatively view Theorem~\\ref{thm:main} as specifying exactly which reward function within each equivalence class the DPO reparameterization selects, that is, the reward function satisfying:\n\\begin{equation}\\label{eq:lag_p}\n     \\sum_{y}\\underbrace{\\piref(y\\mid x)\\exp\\left(\\frac{1}{\\beta}r(x, y)\\right)}_{=\\pi(y\\mid x)\\text{, using Thm.~\\ref{thm:main} reparam.}} = 1,\n\\end{equation}\ni.e., $\\pi(y\\mid x)$ is a valid distribution (probabilities are positive and sum to 1).'

"""You are a quality control assistant. Your task is to review and refine the output of a previous process. The original task was to deconstruct a sentence from a paper into self-contained, atomic facts that ends with their corresponding targets.

### Instructions
Here is a non-exaustive list of things to check for:
- Mathematical notation should *NOT* be written with *unicode* mathematical characters. All math *MUST* be written in LaTeX. Rewrite the fact using the same syntax, formatting, and style of LaTeX as the original sentence and context. Pay attention to details such as whether backslashes should precede parentheses.
- Do not use unnecessary styling commands such as \displaystyle
- Each of the facts should be accurate and consistent with the knowledge presented in the original sentence. 
- One common error during this step is that the facts are not written independently of each other. For instance, Fact 1 may describe a procedure and Fact 2 may refer to it as "this procedure". If you see this, please rewrite the facts so that they are written independently of each other.
- We define the *target* as the last phrase in the sentence. Thus, the fact must end with the specified target. If the fact does not end with the target, consider if the actual last phrase of the fact can be the target, instead. 
- We consider a target to be *valid* if 1) it consists of 1-3 words 2) does not contain any special characters such as parenthesis or mathematical notation and 3) is a piece of information, that passes the "non-trivial" test. This is the "non-trivial" test: if the fact was to be stripped of the target, it should be reasonable to infer the target based on the available context if one has acquired this knowledge. At the same time, it should be impossible to infer the target if one does not have this knowledge. For instance, in the sentence "Chain-of-thought reasoning can be elicited in large language models", the target is large language models. In this case, the target is valid because chain-of-thought reasoning is a property of large language models, making it non-trivial and inferrable from the available context but not completely trivial and obvious to a lay person who does not have this knowledge. 
    - If the target is not valid, discard the fact.

Stick to this template for contextualizing paper-specific details:
- "In the paper '{title}', ..."
- "According to the paper '{title}',..."

This template is not exhaustive, and there are other ways to contextualize the paper-specific details. Use your own judgement to properly contextualize the fact.

Refine the output of the previous process based on these instructions. 

Output the refined list of `(target, rewritten_fact)` pairs. Make sure that any mathematical expressions or notations are written in LaTeX."""

You are a meticulous Quality Control Assistant. Your task is to review, verify, and refine facts that have been extracted from an academic paper. You will be given a list of '(target, rewritten_fact)' pairs and the original context from which they were derived. Your task is to apply a rigorous checklist to each pair, refining it based on a provided quality control checklist or discarding it if it fails critical criteria.

### Overarching Instructions
- Source Grounding: All information in the rewritten fact **MUST** originate from the provided context. Do not add, infer, or correct information using your internal knowledge. Every detail should be traceable back to the source text.
- Preserve Meaning: The refined fact must accurately represent the knowledge in the original sentence without distortion. Strive to use phrasing close to the original text where appropriate, but prioritize clarity and self-containment.
- Discard, Don't Force: If a fact cannot be naturally rephrased to meet the criteria below, or if its target is invalid, it is better to **discard** it than to create an awkward or inaccurate statement.

### Quality Control Checklist

For each '(target, rewritten_fact)' pair, verify the following:

**1. Target Validity:** The target must be valid. A valid target meets all three conditions:
    - **Length:** It consists of 1-3 words.
    - **Formatting:** It contains no special characters, parentheses, or mathematical notation.
    - **Non-Triviality:** It must pass the "non-trivial" test. Ask yourself: "If the target were removed, could a person with knowledge in this specific domain reasonably infer it from the rest of the fact?" The answer should be yes. Conversely, could a layperson without any of the domain knowledge guess it? The answer should be no. The target should be a key piece of information, not just a grammatical stop-word.
    - *Action:* If the target is invalid, **discard the fact**.

**2. Fact-Target Alignment:**
    - The 'rewritten_fact' must end *exactly* with the provided 'target' string.
    - *Action:* If it doesn't, first check if the actual last 1-3 words of the fact could serve as a better, valid target. If so, update the target. If not, **discard the fact**.

**3. Standalone Context:**
    - The 'rewritten_fact' must be fully self-contained and understandable to someone who has **NOT** read the original paper.
    - **Mathematical Context:** If mathematical variables, equations, or theorems are mentioned (e.g., "$r_\\theta(x, y)$"), their full definitions from the context must be included in the fact. Simply referencing an equation number is not sufficient.
    - **Experimental Context:** If the fact describes an experiment, ensure sufficient details are provided (e.g., the specific dataset, the metric used, what is being compared).
    - **Acronyms:** Define acronyms unless they are extremely common or defined and used repeatedly within the provided context itself.
    - *Action:* Add necessary context from the provided text to make the fact fully interpretable on its own. Start with a clarifying phrase like "In the paper '{title}', ..." or "According to the analysis in '{title}', ...".

**4. Fact Independence:**
    - If there are multiple facts, ensure they do not reference each other using pronouns or demonstratives (e.g., "this procedure," "the latter," "it"). Each fact must stand completely on its own.
    - *Action:* Rewrite facts to remove dependencies, duplicating context if necessary.

**5. LaTeX Formatting:**
    - All mathematical expressions and notations **MUST** be written in LaTeX, enclosed in '$' or '$$' delimiters.
    - Do **NOT** use unicode mathematical characters (e.g., use '\\pi', not 'π').
    - Do **NOT** use unnecessary styling commands like '\\displaystyle'.
    - Ensure LaTeX syntax matches the style of the original context (e.g., '( ... )' or '$ ... $').
    - *Action:* Correct any and all formatting errors related to mathematical notation.

### Output Format
After your review, provide a final, clean list of the refined '(target, rewritten_fact)' pairs that have passed all checks.

In [136]:
import concurrent.futures
 
#  First, extract the finalized list of `(target, rewritten_sentence)` pairs from the output of the previous process.

# Then, for each pair of target and fact, carefully perform the following checks.
# - *Target Validity:*  Does the fact properly end with its corresponding target? Does the target only appear once in the fact?
# - *Fact Quality:* Is the writing of the fact natural? Is it fully understandable on its own?

# If any pairs fail the checks, discard them.

# Approach this task carefully, following the step-by-step guidelines provided below and provide your reasoning before the final output.


# Lastly, o

def validate_atomic_facts(row, previous_output):
    """Extract atomic facts from a paragraph using LLM."""
    prompt = {}
    prompt['system'] = """You are a meticulous Quality Control Assistant. Your task is to review, filter, and refine facts that have been extracted from an academic paper. You will be given a list of '(target, rewritten_fact)' pairs and the original context from which they were derived. Your task is to apply a rigorous checklist to each pair, discarding or refining it based on a provided quality control checklist.

### Quality Control Checklist

For each '(target, rewritten_fact)' pair, verify the following:

**1. Target Validity:** The target must be valid. A valid target meets all three conditions:
    - **Length:** It consists of 1-3 words.
    - **Formatting:** It contains no special characters, parentheses, or mathematical notation.
    - **Non-Triviality:** It must pass the "non-trivial" test. Ask yourself: "If the target were removed, could a person with knowledge in this specific domain reasonably infer it from the rest of the fact?" The answer should be yes. Conversely, could a layperson without any of the domain knowledge guess it? The answer should be no. The target should be a key piece of information, not just a grammatical stop-word.
    - *Action:* If the target is invalid, **discard the fact**.

**2. Self-Containment:**
    - The 'rewritten_fact' must be fully self-contained and understandable on its own.
    - **Mathematical Context:** If mathematical variables, equations, or theorems are mentioned (e.g., "$r_\\theta(x, y)$"), their full definitions from the context must be included in the fact. Simply referencing an equation number is not sufficient.
    - **Experimental Context:** If the fact describes an experiment, ensure sufficient details are provided (e.g., the specific dataset, the metric used, what is being compared).
    - **Acronyms:** Define acronyms unless they are extremely common or defined and used repeatedly within the provided context itself.
    - *Action:* If the fact is not interpretable on its own, **discard the fact**.
    
**3. Fact Independence:**
    - If there are multiple facts, ensure they do not reference or rely on each other. For instance, Fact 1 may describe a procedure and Fact 2 may refer to it as "this procedure". If you see this, please go back and rewrite the facts so that they are written independently of each other.
    - *Action:* Rewrite facts, duplicate the context, to remove dependencies.

**4. LaTeX Formatting:**
    - All mathematical expressions and notations **MUST** be written in LaTeX, enclosed in '$' or '$$' delimiters.
    - Do **NOT** use unicode mathematical characters (e.g., use '\\pi', not 'π').
    - Do **NOT** use unnecessary styling commands like '\\displaystyle'.
    - Ensure LaTeX syntax matches the style of the original context (e.g., '( ... )' or '$ ... $').
    - *Action:* Rewrite the facts, correcting any and all formatting errors related to mathematical notation.

### Output Format
After your review, provide a list of the refined '(target, rewritten_fact)' pairs that have passed all checks."""
    context = extract_context_and_sentence(row)
    prompt['user'] = f"""### Context\n{context} ### Output of Previous Process\n{previous_output}"""
    
    response = utils.query_gpt(prompt, model='o4-mini', reasoning_effort='high',system_prompt_included=True)
    return response
    
def format_atomic_facts(previous_output):
    prompt = {}
    prompt['system'] = """Format the final list of targets and their corresponding facts into JSON with the following key. Copy the facts exactly as they are, including any LaTeX formatting.
- "pairs": list of dicts i.e ("target": target, "rewritten_fact": rewritten fact)"""
    prompt['user'] = f"""### List of Facts\n\n{previous_output}"""
    
    response = utils.query_gpt(prompt, model='o4-mini', reasoning_effort='low', return_json=True, system_prompt_included=True)
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        print(f"Failed to parse JSON: {response}")
        return None
    
print(results[5])
filtering = validate_atomic_facts(paper_df_suitable.iloc[86], results[5])
print(filtering)
formatted_facts = format_atomic_facts(filtering)
print(formatted_facts)
    

Below is my step-by-step reasoning and the final contextualized facts with their targets.

Step 1: Fact Extraction  
From the sentence  
“We can alternatively view Theorem 1 as specifying exactly which reward function within each equivalence class the DPO reparameterization selects, that is, the reward function satisfying:  
∑ₙ p_ref(y|x) exp( r(x,y)/β ) = 1,  
i.e., π(y|x) is a valid distribution (probabilities are positive and sum to 1).”  
I extract the following atomic facts (each independent):

1. Theorem 1 specifies exactly which reward function within each equivalence class the DPO reparameterization selects.  
2. The selected reward function satisfies the normalization equation ∑_y p_ref(y|x) exp(1/β r(x,y)) = 1.  
3. The normalization equation implies that π(y|x) is a valid probability distribution (its probabilities are positive and sum to 1).

Step 2: Identifying Targets  
We choose 1–3 word non-trivial targets for each fact:

Fact 1 target: “DPO reparameterization”  
Fact 2

### 4. Filter and Refine Probes

In [168]:
import concurrent.futures
from importlib import reload 
import utils.utils as utils
reload(utils)

def validate_atomic_facts(row):
    """Extract atomic facts from a paragraph using LLM."""
    prompt = {}
    prompt['system'] = """You are a quality control assistant. You will be given the instructions for some task, and numerous attempts of addressing that task. Your task is to review and select the output that best addresses the instructions.

--------------Original Task--------------
"You will be given two inputs, a section of an academic paper for context and a single sentence drawn from that section. Your task is to first extract facts from the sentence. For each fact, rewrite the fact into a self-contained, contextualized fact with an identifiable target at the end. Approach this task step-by-step as outlined below.

### Overarching Instructions
At every step, upheld these core principles:
- Source Grounding: All information *MUST* originate from the provided context or sentence. Do not add, infer, or correct information using your internal knowledge. Every detail should be traceable back to the source text.
- Preserve Meaning: As you write and rewrite the facts, accurately represent the knowledge in the original sentence without distortion. Strive to use phrasing as close as possible to the original text, but prioritize clarity and self-containment.
- Writing Quality: The facts should be written well and clearly so that they are easy to read.

### Step 1: Fact Extraction
Sentences in academic papers often interweave various pieces of knowledge together. Identify and extract the main, atomic facts in the provided sentence; a fact should be what's merely stated by the sentence or a direct implication of the sentence. If there is information that come in the form of "a, b, and c", these three should be kept together in the same fact. The extracted facts must not reference each other and must be self-contained. Each fact should be written independently of the other facts.

One common error during this step is that the facts are not written independently of each other. For instance, Fact 1 may describe a procedure and Fact 2 may refer to it as "this procedure". If you see this, please go back and rewrite the facts so that they are written independently of each other.

### Step 2: Validating Targets
For each fact, take the last phrase in the sentence, and consider whether it is a valid target. We consider a target to be *valid* if 1) the target consists of 1-3 words 2) does not contain any special characters such as parenthesis or mathematical notation and 3) passes the "non-trivial" test. The target does not need to be the most core piece of information; it just needs to be "non-trivial".

This is the "non-trivial" test: if the fact was to be stripped of the target, it should be reasonable to infer the target based on the available context if one has acquired this knowledge. At the same time, it should be impossible to infer the target if one does not have this knowledge. For instance, in the sentence "Chain-of-thought reasoning can be elicited in large language models", the target is large language models. In this case, the target is valid because chain-of-thought reasoning is a property of large language models, making it non-trivial and inferrable from the available context but not completely trivial and obvious to a lay person who does not have this knowledge. 

If the last phrase in the fact is not a valid target, consider how else the fact could have been extracted back in the previous step. If you can't think of any other way to extract the fact, discard the fact.

### Step 3: Contextualize Each Fact
Then, for each fact, add enough context so that the rewritten fact is precise, accurate, self-contained, and interpretable on its own, while still ending with the corresponding target. The rewritten fact doesn't need to be a single sentence; if it's better to break it up, write sentences to set the context, and then a sentence to state the fact. In either case, use the *provided context* to supply whatever information is needed to make the fact independent and standalone. For instance, "Humans and GPT4 agree often with each other" should be clarified into "In the paper '...', humans and GPT4 agreed often with each other during evaluation of DPO" if this notion was in the context of evaluating DPO in an academic paper. Be thorough in your examination to provide the necessary context, but also provide the minimal context necessary as we want to avoid overly verbose sentences. During the contextualization process, please ensure that that the fact still ends the corresponding target. 

Look out for these cases in particular:
- *Math Contextualization:* If there are equations, theorems, or definde variables e.g. $f$ that are being referenced, check the surrounding context and include the full definitions, equation, theorems, and givens that are being referenced to fully contextualize the math. Citing the reference is not enough, but the original definition, equation, theorems, and givens should be included into the fact.
- *Experimental Contextualization:* There are often numerous experiments in a paper, and so supply enough experimental context so that the information is clear and interpretable on its own. 
- *Acronyms*: If the target is an acronym and the acronym appears frequently in the context, feel free to leave it as an acronym without defining it.

For every fact, start with this template for contextualizing:
- "In the paper '{title}', ..."
- "According to the paper '{title}',..."

This is just a starting point and a non-exhaustive template, and you should use your own judgement to supply enough context so that the fact is clear and interpretable on its own.

### Final Instructions
Think carefully and critically through this task, following the step-by-step instructions outlined above, and provide your reasoning. After each step, show the current status of the facts and targets and then provide the final output, listing each fact and its corresponding target. I've provided some demonstrations below that can guide your reasoning and output format. 

### Demonstration 1
Context: "\\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}\n\\subsection{Can DPO scale to real preference datasets?}\nNext, we evaluate fine-tuning performance of DPO on summarization and single-turn dialogue. For summarization, automatic evaluation metrics such as ROUGE can be poorly correlated with human preferences~\citep{stiennon2022learning}, and prior work has found that fine-tuning LMs using PPO on human preferences to provide more effective summaries. We evaluate different methods by sampling completions on the test split of TL;DR summarization dataset, and computing the average win rate against reference completions in the test set."

Sentence: "We evaluate different methods by sampling completions on the test split of TL;DR summarization dataset, and computing the average win rate against reference completions in the test set."

Facts:
- Fact: "In the paper 'Direct Preference Optimization: Your Language Model is Secretly a Reward Model', the authors evaluate DPO’s fine-tuning performance against other methods on summarization by sampling completions on the test split of the dataset named TL;DR summarization", Target: "TL;DR summarization"
- Fact: "In the paper 'Direct Preference Optimization: Your Language Model is Secretly a Reward Model', the fine-tuning performance of DPO and other methods on summarization are evaluated by sampling completions on the test split of the TL;DR summarization dataset and computing the average win rate against the reference completions", Target: "reference completions"

### Demonstration 2
Context: "\\title{Direct Preference Optimization: Your Language Model is Secretly a Reward Model}\nWhile large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training. Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

Sentence: "Existing methods for gaining such steerability collect human labels of the relative quality of model generations and fine-tune the unsupervised LM to align with these preferences, often with reinforcement learning from human feedback (RLHF)."

Facts:
- Fact: "According to the paper 'Direct Preference Optimization: Your Language Model is Secretly a Reward Model', to steer unsupervised language models, existing methods collect human labels", Target: "human labels"
- Fact: "According to the paper 'Direct Preference Optimization: Your Language Model is Secretly a Reward Model', existing methods for steering unsupervised language models collect human labels of the relative quality of model generations", Target: "model generations"
- Fact: "According to the paper 'Direct Preference Optimization: Your Language Model is Secretly a Reward Model', existing methods align unsupervised language models by fine-tuning on human preferences", Target: "preferences"
- Fact: "According to the paper 'Direct Preference Optimization: Your Language Model is Secretly a Reward Model', existing methods for steering unsupervised language models via fine-tuning on human preferences often use reinforcement learning from human feedback", Target: "human feedback""

--------------Reviewing Instructions--------------
We want to finalize the list of `(target, rewritten_sentence)` pairs for this sentence. Based on the instructions that were pasted above, review all the previous attempts of extracting facts, validating targets, and contextualizing facts. Choose a unique, non-overlapping set of the best facts. 

Furthermore, once you have chosen the best facts, PLEASE rewrite the mathematical notation and expressions in proper LaTeX format:
- Use the same LaTeX syntax, spacing, and formatting as the provided context. 
- Do not precede parentheses with a backslash e.g. \\( and make sure to surround algebra, expressions, and LaTeX commands e.g. \\pi with $.
- Do not us any styling commands such as \displaystyle
- Leave numbers alone; don't surround them with $.

Lastly, output the filtered list of `(target, rewritten_sentence)` pairs. Copy the facts exactly as they are."""
    sentence = row['raw_knowledge_statement']
    previous_output_1 = row['raw_extracted_facts_1']
    previous_output_2 = row['raw_extracted_facts_2']
    previous_output_3 = row['raw_extracted_facts_3']
    context = extract_context_and_sentence(row)

    prompt['user'] = f"""### Context\n{context}\n\n### Sentence\n{sentence}\n\n### Output 1 of Previous Process\n{previous_output_1}\n\n### Output 2 of Previous Process\n{previous_output_2}\n\n### Output 3 of Previous Process\n{previous_output_3}"""
    
    response = utils.query_gpt(prompt, model='o4-mini', system_prompt_included=True, reasoning_effort='high')
    return response
    
def format_atomic_facts(previous_output):
    prompt = {}
    prompt['system'] = r"""Format the final list of targets and their corresponding facts into JSON with the following key. Copy the facts exactly as they are, including the LaTeX formatting exactly as it was provided. DO NOT precede parentheses with a backslash \ and make sure to surround algebra or LaTeX commands e.g. \pi with $. Leave numbers alone; don't surround them with $. Only use a single backslash for LaTeX commands e.g. $r_{\phi}(x, y)$.
- "pairs": list of dicts i.e ("target": target, "rewritten_fact": rewritten fact)"""
    prompt['user'] = f"""### List of Facts\n\n{previous_output}"""
    
    response = utils.query_gpt(prompt, model='o4-mini', return_json=True, system_prompt_included=True, reasoning_effort='low')
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        print(f"Failed to parse JSON: {response}")
        return None
    
print(paper_df_suitable.iloc[86]['raw_knowledge_statement'])
print(paper_df_suitable.iloc[86]['raw_extracted_facts_1'])
filtering = validate_atomic_facts(paper_df_suitable.iloc[86])
print(filtering)
formatted_facts = format_atomic_facts(filtering)
print(formatted_facts)
    

We can alternatively view Theorem~\ref{thm:main} as specifying exactly which reward function within each equivalence class the DPO reparameterization selects, that is, the reward function satisfying:
\begin{equation}\label{eq:lag_p}
     \sum_{y}\underbrace{\piref(y\mid x)\exp\left(\frac{1}{\beta}r(x, y)\right)}_{=\pi(y\mid x)\text{, using Thm.~\ref{thm:main} reparam.}} = 1,
\end{equation}
i.e., $\pi(y\mid x)$ is a valid distribution (probabilities are positive and sum to 1).
Fact 1  
According to the paper “Direct Preference Optimization: Your Language Model is Secretly a Reward Model,” Theorem 1 specifies exactly which reward function within each equivalence class is selected by the DPO reparameterization.  
Target: DPO reparameterization  

Fact 2  
According to the paper “Direct Preference Optimization: Your Language Model is Secretly a Reward Model,” the DPO reparameterization selects a reward function satisfying the normalization constraint  
∑ₙ₍y₎ p_ref(y ∣ x) exp(r(x,y)/β) = 1,

In [169]:
def process_row(row):
    """Process a single row for validation."""
    return format_atomic_facts(validate_atomic_facts(row))

# Process all rows in parallel
with concurrent.futures.ThreadPoolExecutor(max_workers=32) as executor:
    validated_results = list(tqdm(
        executor.map(process_row, [row for _, row in paper_df_suitable.iterrows()]),
        total=len(paper_df_suitable)
    ))

# Add results as a new column
paper_df_suitable['validated_atomic_pairs'] = validated_results

print(f"Processed {len([r for r in validated_results if r is not None])} rows with validated pairs")

100%|██████████| 151/151 [04:23<00:00,  1.74s/it]

Processed 151 rows with validated pairs


In [87]:
# Display 10 sample atomic pairs for review
print("Sample Validated Atomic Pairs:")
print("=" * 50)

sample_count = 0
for idx, row in paper_df_suitable.iterrows():
    if row['validated_atomic_pairs'] is not None and sample_count < 100:
        pairs = row['validated_atomic_pairs']
        if pairs:
            print(f"\nRow {idx}:")
            print(f"Original Statement: {row['raw_knowledge_statement']}")
            print(f"Section: {row['section']} - {row['subsection']}")
            print(f"Number of atomic pairs: {len(pairs)}")
            print("Atomic Pairs:")
            for i, pair in enumerate(pairs):
                print(f"  {i+1}. Target: '{pair['target']}'")
                # Fix misspelled key 'rewitten_fact' to 'rewritten_fact'
                if 'rewitten_fact' in pair:
                    pair['rewritten_fact'] = pair.pop('rewitten_fact')
                print(f"     Probe: {pair['statement']}")
            print("-" * 40)
            sample_count += 1
    
    if sample_count >= 100:
        break

if sample_count == 0:
    print("No validated atomic pairs found in the data.")


Sample Validated Atomic Pairs:

Row 0:
Original Statement: While large-scale unsupervised language models (LMs) learn broad world knowledge and some reasoning skills, achieving precise control of their behavior is difficult due to the completely unsupervised nature of their training.
Section: Title/Abstract - No Subsection
Number of atomic pairs: 5
Atomic Pairs:
  1. Target: 'broad world knowledge'
     Probe: In the paper "Direct Preference Optimization: Your Language Model is Secretly a Reward Model", the authors state that large-scale unsupervised language models learn broad world knowledge.
  2. Target: 'reasoning skills'
     Probe: In the paper "Direct Preference Optimization: Your Language Model is Secretly a Reward Model", the authors state that large-scale unsupervised language models learn reasoning skills.
  3. Target: 'precise control'
     Probe: According to the paper "Direct Preference Optimization: Your Language Model is Secretly a Reward Model", what is difficult to ac

### 5. Check Target is truly at end of the sentence -> Split Fact into Fact and Target

In [178]:
# Explode the validated_atomic_pairs into separate rows
rows_for_df = []

for idx, row in paper_df_suitable.iterrows():
    if row['validated_atomic_pairs'] is not None:
        pairs = row['validated_atomic_pairs'].get('pairs', [])
        for pair in pairs:
            new_row = row.to_dict()
            new_row['target'] = pair['target']
            new_row['fact'] = pair['rewritten_fact']
            rows_for_df.append(new_row)

# Create new dataframe with exploded facts
paper_df_with_probes = pd.DataFrame(rows_for_df)

print(f"Exploded {len(paper_df_suitable)} rows with validated pairs into {len(paper_df_with_probes)} individual fact rows")


Exploded 151 rows with validated pairs into 335 individual fact rows


In [179]:
paper_df_with_probes = paper_df_with_probes[['section','subsection', 'section_text','subsection_text', 'raw_knowledge_statement', 'target', 'fact']]

In [180]:
# Fix facts that start with '(' by extracting the rightmost part after target + ','
facts_starting_with_paren = paper_df_with_probes[paper_df_with_probes['fact'].str.startswith('(')]
print(f"Found {len(facts_starting_with_paren)} facts starting with '(' - fixing these...")

for idx, row in facts_starting_with_paren.iterrows():
    target_with_comma = row['target'] + ','
    if target_with_comma in row['fact']:
        rightmost_part = row['fact'].split(target_with_comma)[-1].strip().strip('()')
        # Update the fact in the dataframe
        paper_df_with_probes.loc[idx, 'fact'] = rightmost_part
        print(f"Fixed fact at index {idx}: '{row['fact']}' -> '{rightmost_part}'")

print(f"Fixed {len(facts_starting_with_paren)} facts that started with '('")


Found 0 facts starting with '(' - fixing these...
Fixed 0 facts that started with '('


In [207]:
# Check that facts end with targets (after stripping whitespace and punctuation)
import string

valid_facts = []
filtered_count = 0
dropped_examples = []

for idx, row in paper_df_with_probes.iterrows():
    fact = str(row['fact']).strip()
    target = str(row['target']).strip()
    
    # Remove punctuation from the end of fact for comparison
    fact_cleaned = fact.rstrip(string.whitespace).rstrip('.')
    
    # Check if fact ends with target
    if fact_cleaned.endswith(target):
        valid_facts.append(True)
    else:
        valid_facts.append(False)
        filtered_count += 1
        dropped_examples.append({'fact': fact, 'target': target})

paper_df_with_probes['valid_fact'] = valid_facts

print(f"Filtered out {filtered_count} facts that don't end with their target")
print(f"Remaining valid facts: {len(paper_df_with_probes) - filtered_count}")

# Print some examples that were dropped
if dropped_examples:
    print("\nExamples of dropped fact-target pairs:")
    for i, example in enumerate(dropped_examples[:5]):  # Show first 5 dropped examples
        print(f"  {i+1}. Fact: '{example['fact']}'")
        print(f"     Target: '{example['target']}'")
        print()
        
# Filter to keep only valid facts
paper_df_with_probes = paper_df_with_probes[paper_df_with_probes['valid_fact']].copy()


Filtered out 0 facts that don't end with their target
Remaining valid facts: 279


In [208]:
# Create probe column (fact minus target) using stripped, no punctuation versions
import string

probes = []
cleaned_facts = []
cleaned_targets = []

for idx, row in paper_df_with_probes.iterrows():
    fact = str(row['fact']).strip()
    target = str(row['target']).strip()
    
    # Clean fact and target by removing punctuation from the end
    fact_cleaned = fact.rstrip(string.punctuation + string.whitespace)
    target_cleaned = ' ' + target.rstrip(string.punctuation + string.whitespace)
    
    
    last_index = fact_cleaned.rfind(target_cleaned)
    if last_index != -1:
        probe = fact_cleaned[:last_index].strip()
        probes.append(probe)
        cleaned_facts.append(fact_cleaned)
        cleaned_targets.append(target_cleaned)
    else:
        print(fact)
        print(target)
        raise ValueError(f"Target {target_cleaned} not found in fact {fact_cleaned}")

paper_df_with_probes['probe'] = probes
paper_df_with_probes['fact'] = cleaned_facts
paper_df_with_probes['target'] = cleaned_targets

print(f"Created probe column by removing target from fact")
print(f"Final dataset shape: {paper_df_with_probes.shape}")
#paper_df_with_probes.drop(columns=['valid_fact'], inplace=True)
paper_df_with_probes.head(5)

Created probe column by removing target from fact
Final dataset shape: (279, 9)


,section,subsection,section_text,subsection_text,raw_knowledge_statement,target,fact,probe,valid_fact
0,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...,broad world knowledge,"According to the paper ""Direct Preference Opti...","According to the paper ""Direct Preference Opti...",True
1,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...,some reasoning skills,"According to the paper ""Direct Preference Opti...","According to the paper ""Direct Preference Opti...",True
2,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,While large-scale unsupervised language models...,precise control,"According to the paper ""Direct Preference Opti...","According to the paper ""Direct Preference Opti...",True
3,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,model generations,"According to the paper ""Direct Preference Opti...","According to the paper ""Direct Preference Opti...",True
4,Title/Abstract,No Subsection,\title{Direct Preference Optimization: Your La...,\title{Direct Preference Optimization: Your La...,Existing methods for gaining such steerability...,preferences,"According to the paper ""Direct Preference Opti...","According to the paper ""Direct Preference Opti...",True


### 6. Ensure tokenizing the target separately from the probe is fine

In [209]:
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("allenai/OLMo-2-0425-1B")

contexts = paper_df_with_probes['probe'].tolist()
targets = paper_df_with_probes['target'].tolist()
facts = paper_df_with_probes['fact'].tolist()

print(f"--- Tokenizer and String Consistency Check ---")

string_mismatches = 0
token_mismatches = 0

for i, (context, target, fact) in enumerate(zip(contexts, targets, facts)):
    # --- String Level Check ---
    reconstructed_string = context + target
    string_match = reconstructed_string == fact
    
    if not string_match:
        string_mismatches += 1
        print(f"--- ❌ STRING MISMATCH: Sample #{i} ---")
        print(f"Context: '{context}'")
        print(f"Target:  '{target}'")
        print(f"Fact:    '{fact}'")
        print(f"Reconstructed: '{reconstructed_string}'")
        print(f"Match: {string_match}")
        print("-" * 30)
        continue
    
    # --- Tokenization Level Check ---
    # Method 1: Tokenize the fact directly
    tokenized_fact = tokenizer(fact, add_special_tokens=False, padding=False)['input_ids']
    
    # Method 2: Tokenize parts separately and concatenate
    tokenized_context = tokenizer(context, add_special_tokens=False, padding=False)['input_ids']
    tokenized_target = tokenizer(target, add_special_tokens=False, padding=False)['input_ids']
    tokenized_parts_combined = tokenized_context + tokenized_target

    # --- Comparison ---
    if tokenized_fact != tokenized_parts_combined:
        token_mismatches += 1
        print(f"--- ❌ TOKEN MISMATCH: Sample #{i} ---")
        print(f"Context: '{context}'")
        print(f"Target:  '{target}'")
        print(f"Fact:    '{fact}'")
        print(f"\nTokenizing fact directly:           {tokenized_fact} (Length: {len(tokenized_fact)})")
        print(f"Tokenizing parts and concatenating: {tokenized_parts_combined} (Length: {len(tokenized_parts_combined)})")
        
        # Find differing positions
        min_len = min(len(tokenized_fact), len(tokenized_parts_combined))
        diff_positions = []
        for pos in range(min_len):
            if tokenized_fact[pos] != tokenized_parts_combined[pos]:
                diff_positions.append(pos)
        
        if diff_positions:
            print(f"\nFirst differing positions: {diff_positions[:5]}")
            for pos in diff_positions[:3]:
                fact_token = tokenizer.decode([tokenized_fact[pos]])
                parts_token = tokenizer.decode([tokenized_parts_combined[pos]])
                print(f"  Position {pos}: fact='{fact_token}' vs parts='{parts_token}'")
        
        print("-" * 30)

print(f"\nSummary:")
print(f"String mismatches: {string_mismatches}")
print(f"Token mismatches: {token_mismatches}")
print(f"Total samples: {len(contexts)}")


--- Tokenizer and String Consistency Check ---

Summary:
String mismatches: 0
Token mismatches: 0
Total samples: 279


### 7. Save the Probes

In [210]:
paper_df_with_probes.reset_index(drop=True, inplace=True)
# paper_df_with_probes.drop(columns=['valid_fact'], inplace=True)
paper_df_with_probes.to_csv('../../data/arxiv/DPO_knowledge_probes_v6.csv', index=False)

In [185]:
import pandas as pd
paper_df_with_probes = pd.read_csv('../../data/arxiv/DPO_knowledge_probes_v6.csv')

In [ ]:
# Export facts and targets to txt file for editing
with open('../../data/arxiv/DPO_knowledge_probes_v6_facts.txt', 'w') as f:
    for i, row in paper_df_with_probes.iterrows():
        f.write(row['fact'] + '\n')
        f.write(f"TARGET: {row['target']}\n\n")  # Show target after each fact

print(f"Exported {len(paper_df_with_probes)} facts with targets to DPO_knowledge_probes_v6_facts.txt")
print("Edit the txt file (both facts and targets), then run the next cell to reload")

In [204]:


# Read edited facts and targets back from txt file
with open('../../data/arxiv/DPO_knowledge_probes_v6_facts.txt', 'r') as f:
    content = f.read()
    blocks = [block.strip() for block in content.split('\n\n') if block.strip()]
    edited_facts = []
    edited_targets = []
    for block in blocks:
        lines = block.split('\n')
        fact_lines = []
        target = ""
        for line in lines:
            if line.startswith('TARGET: '):
                target = line[8:]  # Remove 'TARGET: ' prefix
            else:
                fact_lines.append(line)
        edited_facts.append('\n'.join(fact_lines))
        edited_targets.append(target)

# Update the dataframe with edited facts and targets
paper_df_with_probes['fact'] = edited_facts
paper_df_with_probes['target'] = edited_targets

# Save updated dataframe
paper_df_with_probes.to_csv('../../data/arxiv/DPO_knowledge_probes_v6.csv', index=False)

print(f"Reloaded {len(edited_facts)} facts and targets from txt file and updated v6 CSV")


Reloaded 279 facts and targets from txt file and updated v6 CSV


Reloaded 279 facts from txt file and updated v6 CSV


In [188]:
grouped = paper_df_with_probes.groupby('raw_knowledge_statement')
group_88 = list(grouped)[88]  # 88th group (0-indexed)
raw_statement, group_df = group_88

print(f"{extract_context_and_sentence(group_df.iloc[0])}")
print(f"88th group - Raw knowledge statement:")
print(f"{raw_statement}")
print(f"\nFacts generated from this statement:")
for i, fact in enumerate(group_df['fact']):
    print(f"  {i}: {fact}")


In this section, we give further interpretation of the DPO method, provide theoretical backing, and relate advantages of DPO to issues with actor critic algorithms used for RLHF (such as PPO~\cite{schulman2017proximal}).

\label{sec:theory}

\subsection{Your Language Model Is Secretly a Reward Model} DPO is able to bypass both fitting an explicit reward and performing RL to learn the policy using a single maximum likelihood objective. Note the optimization objective Eq. \ref{eq:main_eq} is equivalent to a Bradley-Terry model with a reward parameterization $r^*(x, y) = \beta \log\frac{\pi^*_\theta(y \mid x)}{\piref(y \mid x)}$ and we optimize our parametric model $\pi_{\theta}$, equivalently to the reward model optimization in Eq. \ref{eq:reward_model} under the change of variables. In this section we will build the theory behind this reparameterization, show that it does not constrain the class of learned reward models, and allows for the exact recovery of the optimal policy. We begin 

\label{sec:DPO}

Motivated by the challenges of applying reinforcement learning algorithms on large-scale problems such as fine-tuning language models, our goal is to derive a simple approach for policy optimization using preferences directly. Unlike prior RLHF methods, which learn a reward and then optimize it via RL, our approach leverages a particular choice of reward model parameterization that enables extraction of its optimal policy in closed form, without an RL training loop. 
As we will describe next in detail, our key insight is to leverage an analytical mapping from reward functions to optimal policies, which enables us to transform a loss function over reward functions into a loss function over policies.
This change-of-variables approach avoids fitting an explicit, standalone reward model, while still optimizing under existing models of human preferences, such as the Bradley-Terry model. In essence, the policy network represents both the language model and the (implicit) reward.

\textbf{Deriving the DPO objective.} We start with the same RL objective as prior work, Eq.~\ref{eq:RL}, under a general reward function $r$. Following prior work~\citep{peters2007reinforcement, peng2019advantage, korbak2022reinforcement, go2023aligning}, it is straightforward to show that the optimal solution to the KL-constrained reward maximization objective in Eq.~\ref{eq:RL} takes the form:
\begin{equation}\label{eq:op_policy}
    \pi_r(y\mid x) = \frac{1}{Z(x)}\piref(y\mid x)\exp\left(\frac{1}{\beta}r(x, y)\right),
\end{equation}%
where $Z(x) =\sum_{y}\piref(y\mid x)\exp\left(\frac{1}{\beta}r(x, y)\right)$ is the partition function. See Appendix \ref{app:derivation1} for a complete derivation. Even if we use the MLE estimate $r_{\phi}$ of the ground-truth reward function $r^*$, it is still expensive to estimate the partition function $Z(x)$ \citep{korbak2022reinforcement, go2023aligning}, which makes this representation hard to utilize in practice. However, we can rearrange Eq.~\ref{eq:op_policy} to express the reward function in terms of its corresponding optimal policy $\pi_r$, the reference policy $\piref$, and the unknown partition function $Z(\cdot)$. Specifically, we first take the logarithm of both sides of Eq.~\ref{eq:op_policy} and then with some algebra we obtain:
\begin{equation}\label{eq:main_eq}
    r(x,y) =\beta \log \frac{\pi_r(y\mid x)}{\piref(y\mid x)} + \beta \log Z(x).
\end{equation}
We can apply this reparameterization to the ground-truth reward $r^*$ and corresponding optimal model $\pi^*$. Fortunately, the Bradley-Terry model depends only on the difference of rewards between two completions, i.e., ${p^*(y_1 \succ y_2 \mid x) = \sigma(r^*(x, y_1) - r^*(x, y_2))}$. Substituting the reparameterization in Eq.~\ref{eq:main_eq} for $r^*(x,y)$ into the preference model Eq.~\ref{eq:bradley-terry}, the partition function cancels, and we can express the human preference probability in terms of only the optimal policy $\pi^*$ and reference policy $\piref$. Thus, the optimal RLHF policy $\pi^*$ under the Bradley-Terry model satisfies the preference model:
\begin{equation}\label{eq:objective}
    p^*(y_1\succ y_2 \mid x)=\frac{1}{1 + \exp\left(\beta \log \frac{\pi^*(y_2\mid x)}{\piref(y_2\mid x)} - \beta \log \frac{\pi^*(y_1\mid x)}{\piref(y_1\mid x)}\right)}
\end{equation}
The derivation is in Appendix~\ref{app:derivation2}. While Eq.~\ref{eq:objective} uses the Bradley-Terry model, we can similarly derive expressions under the more general Plackett-Luce models~\citep{plackett1975analysis, luce2012individual}, shown in Appendix~\ref{app:plackett_luce_models}.

Now that we have the probability of human preference data in terms of the optimal policy rather than the reward model, we can formulate a maximum likelihood objective for a parametrized policy $\pi_\theta$. Analogous to the reward modeling approach (i.e. Eq.~\ref{eq:reward_model}), our policy objective becomes:
\begin{equation}\label{eq:optimum_model}
    \mathcal{L}_\text{DPO}(\pi_{\theta}; \piref) = -\mathbb{E}_{(x, y_w, y_l)\sim \mathcal{D}}\left[\log \sigma \left(\beta \log \frac{\pi_{\theta}(y_w\mid x)}{\piref(y_w\mid x)} - \beta \log \frac{\pi_{\theta}(y_l\mid x)}{\piref(y_l\mid x)}\right)\right].
\end{equation}
This way, we fit an implicit reward using an alternative parameterization, whose optimal policy is simply $\pi_\theta$. Moreover, since our procedure is equivalent to fitting a reparametrized Bradley-Terry model, it enjoys certain theoretical properties, such as consistencies under suitable assumption of the preference data distribution \cite{bong2022generalized}. In Section~\ref{sec:theory}, we further discuss theoretical properties of DPO in relation to other works.

\textbf{What does the DPO update do?} For a mechanistic understanding of DPO, it is useful to analyze the gradient of the loss function $\mathcal{L}_\text{DPO}$. The gradient with respect to the parameters $\theta$ can be written as:
\begin{multline*}\label{eq:gradient}
    \nabla_\theta \mathcal{L}_\text{DPO}(\pi_\theta;\piref) = \\ -\beta\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \bigg[\underbrace{\sigma(\hat{r}_\theta(x, y_l) - \hat{r}_\theta (x, y_w))}_\text{higher weight when reward estimate is wrong}\bigg[\underbrace{\nabla_\theta\log \pi(y_w \mid x)}_\text{increase likelihood of $y_w$} - \underbrace{\nabla_\theta\log\pi(y_l \mid x)}_\text{decrease likelihood of $y_l$}\bigg]\bigg],
\end{multline*}
where $\hat{r}_\theta(x, y) = \beta \log \frac{\pi_\theta(y \mid x)}{\piref(y \mid x)}$ is the reward implicitly defined by the language model $\pi_\theta$ and reference model $\piref$ (more in Section~\ref{sec:theory}). Intuitively, the gradient of the loss function $\mathcal{L}_\text{DPO}$ increases the likelihood of the preferred completions $y_w$ and decreases the likelihood of dispreferred completions $y_l$. Importantly, the examples are weighed by how much higher the implicit reward model $\hat{r}_\theta$ rates the dispreferred completions, scaled by $\beta$, i.e, how incorrectly the implicit reward model orders the completions, accounting for the strength of the KL constraint. Our experiments suggest the importance of this weighting, as a na\"ive version of this method without the weighting coefficient can cause the language model to degenerate (Appendix Table~\ref{tab:unlikelihood_generations}).

\textbf{DPO outline.} 
The general DPO pipeline is as follows: 1) Sample completions $y_1, y_2 \sim \piref(\cdot \mid x)$ for every prompt $x$, label with human preferences to construct the offline dataset of preferences $\mathcal{D} = \{x^{(i)}, y_w^{(i)}, y_l)^{(i)}\}_{i=1}^N$ and 2) optimize the language model $\pi_\theta$ to minimize $\mathcal{L}_\text{DPO}$ for the given $\piref$ and $\mathcal{D}$ and desired $\beta$. 
In practice, one would like to reuse preference datasets publicly available, rather than generating samples and gathering human preferences. Since the preference datasets are sampled using $\pisft$, we initialize $\piref = \pisft$ whenever available. However, when $\pisft$ is not available, we initialize $\piref$ by maximizing likelihood of preferred completions ${(x, y_w)}$, that is, ${\piref = \argmax_{\pi}\mathbb{E}_{x, y_w \sim \mathcal{D}}\left[\log \pi(y_w \mid x)\right]}$. This procedure helps mitigate the distribution shift between the true reference distribution which is unavailable, and $\piref$ used by DPO. Further details related to the implementation and hyperparameters can be found in Appendix~\ref{app:implementation}.


88th group - Raw knowledge statement:
Since the preference datasets are sampled using $\pisft$, we initialize $\piref = \pisft$ whenever available.

Facts generated from this statement:
  0: In the Direct Preference Optimization (DPO) pipeline, the reference policy $\pi_{\text{ref}}$ is initialized to the supervised fine-tuned policy
  1: To match how the preference data were generated, DPO initializes its reference policy $\pi_{\text{ref}}$ to the supervised fine-tuned policy
  2: Direct Preference Optimization reuses its sampling model by making the reference policy identical to the supervised fine-tuned policy

Context: The general DPO pipeline is as follows: 1) Sample completions $y_1, y_2 \sim \piref(\cdot \mid x)$ for every prompt $x$, label with human preferences to construct the offline dataset of preferences $\mathcal{D} = \{x^{(i)}, y_w^{(i)}, y_l)^{(i)}\}_{i=1}^N$ and 2) optimize the language model $\pi_\theta$ to minimize $\mathcal{L}_\text{DPO}$ for the given $\piref$ and $\mathcal{D}$ and desired $\beta$. The base reference policy can be referred to as $\piref$, and the initial SFT model as $\pisft$.

In practice, one would like to reuse preference datasets publicly available, rather than generating samples and gathering human preferences. Since the preference datasets are sampled using $\pisft$, we initialize $\piref = \pisft$ whenever available.

Sentence: Since the preference datasets are sampled using $\pisft$, we initialize $\piref = \pisft$ whenever available.

Targets: SFT model ($\pisft$)

Facts:
- In the Direct Preference Optimization (DPO) pipeline, the base reference policy is initialized to the available SFT model.



In [48]:
print("All facts in paper_df_with_probes:")
for i, fact in enumerate(paper_df_with_probes['fact']):
    print(f"{i}: {fact}")


All facts in paper_df_with_probes:
0: To guide the behavior of unsupervised language models, current methods rely on collecting human labels
1: Existing techniques for aligning large-scale unsupervised LMs depend on acquiring human labels
2: To enable steerability in unsupervised language models, researchers gather human labels
3: Existing alignment approaches rely on human judgments to rate the relative quality of model generations
4: To evaluate steerability, current methods gather human labels for the relative quality of model generations
5: In steering unsupervised language models, methods obtain human assessments of the relative quality of model generations
6: Reinforcement learning from human feedback (RLHF) for aligning large unsupervised language models with human preferences is a complex procedure
7: To align a large unsupervised LM with human preferences, RLHF employs a complex procedure
8: The method of RLHF for matching model outputs to human preferences involves a complex 

### Appendix A: Examining Tokenization of "X." vs X"

In [9]:
# Check tokenization of "Z(x)" variants
import transformers

# Load the tokenizer (using the same one as in your model)
tokenizer = transformers.AutoTokenizer.from_pretrained("allenai/OLMo-2-0425-1B")

# Test strings
test_strings = ["Z(x)", "Z(x).", "Z(x) )", "\\beta \\log Z(x)"]

print("Checking tokenization of Z(x) variants:")
for test_str in test_strings:
    tokens = tokenizer.tokenize(test_str)
    token_ids = tokenizer.encode(test_str, add_special_tokens=False)
    print(f"'{test_str}' -> tokens: {tokens} -> ids: {token_ids}")

# Check if "Z(x)" is a common prefix
zx_tokens = tokenizer.tokenize("Z(x)")
zx_ids = tokenizer.encode("Z(x)", add_special_tokens=False)

print(f"\nBase 'Z(x)' tokens: {zx_tokens} -> ids: {zx_ids}")

for test_str in ["Z(x).", "Z(x) )"]:
    test_tokens = tokenizer.tokenize(test_str)
    test_ids = tokenizer.encode(test_str, add_special_tokens=False)
    
    # Check if Z(x) tokens are a prefix
    is_prefix = len(zx_ids) <= len(test_ids) and test_ids[:len(zx_ids)] == zx_ids
    print(f"'{test_str}' has 'Z(x)' as prefix: {is_prefix}")


Checking tokenization of Z(x) variants:
'Z(x)' -> tokens: ['Z', '(x', ')'] -> ids: [57, 2120, 8]
'Z(x).' -> tokens: ['Z', '(x', ').'] -> ids: [57, 2120, 570]
'Z(x) )' -> tokens: ['Z', '(x', ')', 'Ġ)'] -> ids: [57, 2120, 8, 883]
'\beta \log Z(x)' -> tokens: ['\\', 'beta', 'Ġ\\', 'log', 'ĠZ', '(x', ')'] -> ids: [59, 19674, 1144, 848, 1901, 2120, 8]

Base 'Z(x)' tokens: ['Z', '(x', ')'] -> ids: [57, 2120, 8]
'Z(x).' has 'Z(x)' as prefix: False
'Z(x) )' has 'Z(x)' as prefix: True


In [31]:
# Check tokenization of the mathematical expression
test_expression = r"\beta \log \frac{\pi_r(y\mid x)}{\piref(y\mid x)}$."

print(f"Testing tokenization of: {test_expression}")
tokens = tokenizer.tokenize(test_expression)
token_ids = tokenizer.encode(test_expression, add_special_tokens=False)
print(f"Tokens: {tokens}")
print(f"Token IDs: {token_ids}")
print(f"Number of tokens: {len(tokens)}")


Testing tokenization of: \beta \log \frac{\pi_r(y\mid x)}{\piref(y\mid x)}$.
Tokens: ['\\', 'beta', 'Ġ\\', 'log', 'Ġ\\', 'frac', '{\\', 'pi', '_r', '(y', '\\', 'mid', 'Ġx', ')}', '{\\', 'pire', 'f', '(y', '\\', 'mid', 'Ġx', ')}', '$.']
Token IDs: [59, 19674, 1144, 848, 1144, 38118, 36802, 2554, 1745, 7166, 59, 16497, 865, 9317, 36802, 23772, 69, 7166, 59, 16497, 865, 9317, 13244]
Number of tokens: 23


# Appendix B: Old Prompts


You will be given two inputs, a full paragraph for context and a single sentence drawn from that paragraph. Your task is to, first, identify the key, central information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target placed at the end of the sentence. Approach this task step-by-step as outlined below.

### Step 1: Target Extraction
First, identify the key central information of a sentence i.e. the targets. For instance, in the sentence, "Chain-of-thought reasoning can be readily elicited in large  language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key, central information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Linguistically, the central information is often the main predicate e.g. "elicited" or, for copular clauses, the subjective complement. For more complex clauses, additional targets can be found elsewhere such as the object of the preposition. Apply this reductionary thought process and these linguistic heuritiscs that I've demonstrated above to identify the targets. A target must be only 1-2 words and should not be a mathematical expression in LaTeX. Err on the side of extracting more targets than less.

### Step 2: Rewriting Facts with Target
Second, for each target, we will rewrite the sentence to become a 1) fully self-contained and atomic fact that 2) ends with the target. The fact should only contain information *relevant* to the target.
Use the surrounding paragraph to supply whatever context is needed to make the fact standalone. The fact should be clear on its own and the knowledge should be self-contained, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the paragraph is about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that makes the information true. Then, ensure that contextualized, self-contained, atomic facts ends with the targets. If this is impossible for a target, then discard the target and output "NA" for the target.

### Step 3: Reflect and Refine

There are a few cases to look out for in particular. If there are mathematical expressions that are being referenced, check the surrounding context and include the original definitions and givens to fully contextualize the sentence so that it can stand alone. If the references cannot be found, discard all probes for this sentence and output "NA". For experimental details, ensure that the details are fully contextualized since experiments, in particular, only make sense in the context and scope of the paper. 

Furthermore, for each fact, check the following:
* **Target Placement:** Does the sentence actually end with the target word(s)? 
* **Self-Contained:** Is the sentence **fully understandable on its own**? 
* **Accuracy:** Does the rewritten fact **preserve the exact meaning** of the original without distortion? Note some facts preserve only parts of the original sentence; this is fine. It should be still be accurate to the original meaning.
* **Fact Quality:** Is the fact clearly obvious from the sentence? Is the construction and writing of the fact natural and unforced? Does the fact flow naturally to end with the target?

Refine the facts to satisfy these conditions. If it's difficult to refine, feel free to discard them since we were liberal with our first step of extracting more targets than less.

### Final Instructions

I've provided some demonstrations below which are some concrete examples that can guide your final output format. Before outputting the final format, think carefully through this task, following the step-by-step instructions outlined above, and provide your reasoning before the final extraction.



---

You will be given two inputs, a full paragraph for context and a single sentence drawn from that paragraph. Your task is to, first, identify the key, central information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target placed at the end of the sentence. Approach this task step-by-step as outlined below.

First, identify the keym central information of a sentence i.e. the target. For instance, in the sentence, "Chain-of-thought reasoning can be readily elicited in large  language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key, central information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Linguistically, the central information is often found in the main predicate e.g. "elicited" or, for copular clauses, the subjective complement. For complex clauses, additional targets can be found elsewhere such as the object of the preposition. Apply this reductionary thought process and these linguistic heuritiscs that I've demonstrated above to identify the targets. A target must be only 1-2 words. There are often multiple targets, but make sure to select the linguistically valid targets i.e. the sentence can be naturally paraphrased to end with the target.

Second, for each target, we will rewrite the sentence to become 1) fully self-contained and atomic and 2) end with the target. Use the surrounding paragraph to supply whatever context is needed to make the sentence standalone. The sentence should be clear on its own and the knowledge should be self-contained, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the paragraph is about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that makes the information true. Then, ensure that this contextualized, self-contained, atomic sentence ends with the target. If this is impossible, then discard this target.

I've provided some demonstrations below which are some concrete examples that can guide your thought process and output format.

----

prompt['system'] = """You will be given two inputs, a full paragraph for context and a single sentence drawn from that paragraph. Your task is to, first, identify the key information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target at the end of the sentence. Approach this task step-by-step as outlined below.

First, identify the key information of a sentence i.e. the target. For instance, in the sentence, "Finally, chain-of-thought reasoning can be readily elicited in sufficiently large off-the-shelf language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Apply this same reductionary thought process that I've demonstrated here to identify the targets. Often times, there are multiple valid targets, but only select the targets such that the sentence can be paragraphased to end with the target.

Second, for each target, we will rewrite the sentence to become 1) atomic and 2) end with the target. Use the surrounding paragraph to supply whatever context is needed to make the sentence standalone. The sentence should be clear on its own and the knowledge should be self-contained, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the paragraph is about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that the information is true for. Then, ensure that this contextualized, self-contained, atomic sentence ends with the target. If this is impossible, then discard this target.

----


Write each fact as one declarative sentence whose final 1–3 words form the target capturing the key information; the probe is the same sentence with the final target span removed and should read naturally as a cloze. Prefer precise, contentful targets; ensure the last words are exactly the target; avoid cross-references to other facts; preserve the original meaning; extract at most three facts per sentence; and present results in the demonstrated Probe/Target format.

    prompt['system'] = """You will be given two inputs: (1) a full paragraph for context and (2) a single sentence drawn from that paragraph. Your task is to rewrite that sentence into 1–3 self-contained, atomic facts that can stand entirely on their own as probe–target pairs. Use the paragraph only to supply whatever context is needed to make each fact standalone; explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers. Also, do not introduce information not entailed by the sentence. Write each fact as one declarative sentence whose final 1–3 words form the target capturing the key information; the probe is the same sentence with the final target span removed and should read naturally as a cloze. Prefer precise, contentful targets; ensure the last words are exactly the target; avoid cross-references to other facts; preserve the original meaning; extract at most three facts per sentence; and present results in the demonstrated Probe/Target format.

-----
"You will be given two inputs, a full paragraph for context and a single sentence drawn from that paragraph. Your task is to, first, identify the key, central information in that sentence given the surrounding context; this will be the target. For each target, rewrite that sentence into a self-contained, atomic fact with the target placed at the end of the sentence. Approach this task step-by-step as outlined below.

First, identify the keym central information of a sentence i.e. the target. For instance, in the sentence, "Chain-of-thought reasoning can be readily elicited in large  language models simply by including examples of chain of thought sequences into the exemplars of few-shot prompting", I would argue the key, central information is "including examples" and "elicited" since the core of this sentence is about how chain-of-thought can be "elicited" by "including examples". Linguistically, the central information is often found in the main predicate e.g. "elicited" or, for copular clauses, the subjective complement. For complex clauses, additional targets can be found elsewhere such as the object of the preposition. Apply this reductionary thought process and these linguistic heuritiscs that I've demonstrated above to identify the targets. A target must be only 1-2 words. There are often multiple targets, but make sure to select the linguistically valid targets i.e. the sentence can be naturally paraphrased to end with the target.

Second, for each target, we will rewrite the sentence to become 1) fully self-contained and atomic and 2) end with the target. Use the surrounding paragraph to supply whatever context is needed to make the sentence standalone. The sentence should be clear on its own and the knowledge should be self-contained, meaning the context needs to be explicitly clarified (e.g. "Humans and GPT4 agree often with each other" should be clarified into "Regarding the evaluation of DPO, Humans and GPT4 agree often with each other" if the paragraph is about evaluating DPO). Explicitly resolve unclear pronouns, name entities, define acronyms on first use, include necessary scope, conditions, and qualifiers that makes the information true. Then, ensure that this contextualized, self-contained, atomic sentence ends with the target. If this is impossible, then discard this target."

Adjust this prompt above to request the following: "take the output and double check if the probes have properly satisfied the conditions or can be even improved i.e. can be formatted in a more natural way, stays true to the original meaning of the sentence, fully contextualizes the sentence to be self-contained, the target is non-trivial and a central part of the sentence, and is rewritten to naturally place the target at the end. Please filter out probes that that don't meet these conditions and refine the remaining probes."

--

reflection_prompt = """
You are a quality control assistant. Your task is to review and refine the output of a previous process. The original task was to deconstruct a sentence from a paragraph into self-contained, atomic facts.

## Background: The Original Task

The process you are reviewing was as follows:
1.  **Input:** A full paragraph for context and a single sentence from that paragraph.
2.  **Step 1: Identify Targets.** The primary goal was to identify the key, central information of the sentence, called a "target".
    * A valid **target** must be **1-2 words** long.
    * A target must **not** be a mathematical expression (e.g., in LaTeX).
    * Linguistically, a target is often the main predicate (verb), the subjective complement, or even the object of the preposition.
    * Multiple targets may exist for a sentence.
3.  **Step 2: Rewrite into Atomic Probes.** For each valid target, the original sentence was to be rewritten into a self-contained, atomic sentence (a "probe").
    * The probe must be **fully self-contained**, meaning it's completely understandable without the original paragraph. This requires adding context, defining acronyms, and resolving pronouns.
    * Crucially, the probe **must end exactly with the target**.
    * If a sentence could not be rewritten to end with a target, that target was to be discarded.

## Your Task Now: Self-Reflection and Correction

Critically review a provided list of `(target, rewritten_sentence)` pairs against the original rules. For each pair, perform the following checks.

#### Check 1: Target Validation
* **Word Count:** Is the target **exactly one or two words**?
* **Content:** Does the target contain any mathematical notation? (It shouldn't).

If a target fails any of these checks, the entire pair is **invalid**.

#### Check 2: Rewritten Sentence Validation
* **Target Placement:** Does the sentence actually end with the target word(s)? This is the most important rule.
* **Self-Contained:** Is the sentence **fully understandable on its own**? Check for undefined acronyms, unresolved pronouns (like 'it', 'they', 'this'), or ambiguous terms that need clarification from the original context.
* **Clarity:** Does the sentence clearly express **a clear fact**?
* **Accuracy:** Does the rewritten sentence **preserve the exact meaning** of the original without distortion?

If the rewritten sentence fails any of these checks, the entire pair is **invalid**.

### Final Instructions

Based on your review:
1.  **Filter:** Discard any pair that fails the validation checks above. 
2.  **Refine:** For pairs that are valid but could be improved, edit the rewritten sentence.
3.  **Format:** Format the final, filtered, and refined list of targets and their corresponding atomic sentences into JSON with the following keys.

"pairs": list of (target, rewritten probe) 
"""
